# Lakelink Fiber PRI: v4 candidate writers (@data-engineer)

This is the versioned writer for the v4 shadow-build objects that @data-engineer owns (task P37-DE-V4-WRITERS-CHECKIN). All write targets are `*_v4_candidate` objects in `cdm_tmforum.provable_resilience_intelligence`. It never writes a v3 FROZEN object. v3 inputs are read only through their FREEZE-v3 pins (`VERSION AS OF`).

| Cell | Object | Rule / task |
| --- | --- | --- |
| 3 | `physical_resource_span_v4_candidate` | B1: BUG-DP-068 BB to BBS segmentation, plus the BUG-QA-075 planar chord split at equal haversine arc length |
| 4 | `physical_resource_conduit_v4_candidate` | A1: BUG-QA-066 geodesic length; P35 coverage class; B2: CDT-BBR-* route conduits |
| 5 | `hazard_span_intersection_v4_candidate` | BUG-QA-065 line distance and BUG-QA-064 rounding to 4 dp, from span@11 |
| 5b | `hazard_span_intersection_v4_candidate` | BUG-QA-073 + BUG-DP-071 (@data-planner): BBS-* re-derivation, SPAN dedup, DROP new out-of-radius pairs, gates A-F. Needs widget `qa073_write_approved=true` as well |
| 6 | `resource_alarm_v4_candidate`, `resource_performance_measure_v4_candidate` | B5: BUG-QA-072 hash-rank round-robin remap |
| 7 | `mv_mrc_protected_v4_candidate`, `hazard_proximity_count_v4_candidate` | Metric view DDL, verbatim |
| 7b | `proactive_action_v4_candidate` | P52: proactive_action@15 carried (BUG-QA-128: span_id corrected to working hop-1; BUG-QA-132: BB→BBS remap uses the P54 orientation rule, so it equals service_path_v4 working hop 1) plus ACT-RULE-v1 on `resilience_score_snapshot_v4_candidate`; gates P52-G1..G4, BUG-QA-128. Needs widget `p52_write_approved=true` (after P36-ML rescore) |
| 7c | `service_path_v4_candidate` | P54 (B3, P53 spec): service_path@11 with every BB-* hop expanded into oriented BBS-* segments (PREV/NEXT_HOP_JUNCTION, NEAREST_ENDPOINT_TO_A_END), hop_sequence renumbered; gates P54-G1..G8 (1,655,121 rows), plus the independent BUG-QA-132 gate against cell 7b output. Needs widget `p54_write_approved=true` as well (cutover window). Count DISTINCT `v3_span_id` for hop counts |
| 8 | Gates | Recompute vs live, 0 diffs expected |

**Modes.** The `write_enabled` widget defaults to `false`. In that mode (dry run), every object is recomputed into a temp view and diffed against the live table, and nothing is written. With `true`, tables are overwritten with `CREATE TABLE IF NOT EXISTS` followed by `INSERT OVERWRITE`, which keeps column comments. Metric views are re-created. Writes are only allowed inside an approved v4 write window (DEC-PRI-V4-SHADOW-BUILD). During the freeze, run in dry-run mode only.

**Pins.** `physical_resource_span@20`, `physical_resource_span@11` (HSI only), `physical_resource_conduit@5`, `hazard_span_intersection@22`, `geographic_hazard_event@34`, `resource_alarm@12`, `resource_performance_measure@8`. Build timestamps are fixed constants, so a rerun reproduces the tables exactly.

**Known limit.** The HSI candidate is still keyed on v3 `BB-*` route ids (18,495 rows). Its rebuild on the BBS ids is BUG-QA-073 (@data-planner), in cell 5b. Once 5b has been written, cell 5's diff against live is expected to fail, because 5b supersedes it.

In [0]:
import math
import re

import numpy as np
import pandas as pd

dbutils.widgets.dropdown("write_enabled", "false", ["false", "true"], "Write v4 candidate objects")
WRITE_ENABLED = dbutils.widgets.get("write_enabled") == "true"

S = "cdm_tmforum.provable_resilience_intelligence"
SPAN_V4 = f"{S}.physical_resource_span_v4_candidate"
CONDUIT_V4 = f"{S}.physical_resource_conduit_v4_candidate"
HSI_V4 = f"{S}.hazard_span_intersection_v4_candidate"
ALARM_V4 = f"{S}.resource_alarm_v4_candidate"
PERF_V4 = f"{S}.resource_performance_measure_v4_candidate"
MV_MRC_V4 = f"{S}.mv_mrc_protected_v4_candidate"
MV_HPC_V4 = f"{S}.hazard_proximity_count_v4_candidate"

# FREEZE-v3 read pins (reads only). P65 / DEC-PRI-V4-PIN-FINGERPRINT-GATES: the numbers are INFORMATIONAL labels.
# Every v3 read goes through v3src(table, pin), which is gated on a pin-free fingerprint, not on Delta time travel.
PIN_SPAN = 20          # physical_resource_span@20 (identical data to @11; 12-20 are comment/property changes)
PIN_SPAN_HSI = 11      # physical_resource_span@11, the pin the HSI candidate was measured on
PIN_CONDUIT = 5        # physical_resource_conduit@5
PIN_HSI = 22           # hazard_span_intersection@22
PIN_HAZARD = 34        # geographic_hazard_event@34
PIN_ALARM = 12         # resource_alarm@12
PIN_PERF = 8           # resource_performance_measure@8

# Pin-free equivalence fingerprints (DE S60/S61, BUG-QA-098): (row count, v1, v2). v1 = sum of conv(substr(sha2(
# concat_ws('|', every column cast to string, table column order), 256), 1, 15), 16, 10) (NULL-skip). v2 = same with
# every column COALESCEd to '<NULL>' first (NULL-token, DEC-PRI-V4-PIN-FINGERPRINT-GATES amendment v2), so a NULL that
# moves between columns is detected. Manifest pins equal freeze_pin_fingerprint_gate (fp_row_count, fp_sha2_v1_null_skip,
# fp_sha2_v2_null_token); span@20, PA@14 and snap@22 recomputed by DE S64 2026-10-30. Gate = all 3 components.
# Verified 2026-10-30 against VERSION AS OF where still readable.
# span@11 and conduit@5 are already past retention; their data equals current because every later commit is
# CHANGE COLUMN / SET TBLPROPERTIES (DESCRIBE HISTORY).
PIN_FP = {
    ("physical_resource_span", 20): (10092, 5774878037168954794297, 5774878037168954794297),
    ("physical_resource_span", 11): (10092, 5774878037168954794297, 5774878037168954794297),
    ("physical_resource_conduit", 5): (1969, 1116608226731745840669, 1116608226731745840669),
    ("hazard_span_intersection", 22): (173323, 99977665921217685096292, 99939384313133177266648),
    ("geographic_hazard_event", 34): (57399, 33146307863512455602412, 33081860638141479101998),
    ("resource_alarm", 12): (2016881, 1161879536387220729156356, 1162079437945972189093170),
    ("resource_performance_measure", 8): (5046315, 2909384958102082528408557, 2909384958102082528408557),
    ("proactive_action", 15): (11276, 6519951217275536975410, 6518768536880426530767),
    ("proactive_action", 14): (10801, 6262085847865777996727, 6246840381215060444268),
    ("resilience_score_snapshot", 22): (8000, 4649660861034310545477, 4649660861034310545477),
    ("service_path", 11): (54572, 31496287517042976573575, 31496287517042976573575),
    ("hazard_span_intersection_v4_candidate", 33): (200089, 115057759381094871266868, 115003341663508063836801),
}
assert all(len(v) == 3 for v in PIN_FP.values()), "BUG-QA-098: every PIN_FP entry must be (n, v1, v2)"
# Pin-free reconstruction of a pin that is NOT equal to current data.
# proactive_action@15 = @14 + one WRITE append of the 475 ACT-V22-* rows, so @14 = current minus those rows (verified diff 0/0).
PIN_FILTER = {("proactive_action", 14): "action_id NOT LIKE 'ACT-V22-%'"}
_V3SRC = {}

# P67-DE-RUNBOOK-RESTORE-VERSION / BUG-QA-095: the time-travel fallback reads demo_freeze_manifest.restore_version
# (the readable Delta version), NOT the delta_version lineage pin. PIN_* numbers above are fingerprint keys and
# lineage labels only. span@11 and conduit@5 are already past retention; their restore_versions (20, 7) are readable.
RESTORE_VERSION = {r.object_name: int(r.restore_version) for r in spark.sql(f"""
    SELECT object_name, restore_version FROM {S}.demo_freeze_manifest
    WHERE manifest_version IN ('PRI-DEMO-FREEZE-v3', 'PRI-DEMO-FREEZE-v4-DRAFT')
      AND freeze_status IN ('FROZEN', 'V4_CANDIDATE', 'V4_CANDIDATE_WIP') AND restore_version IS NOT NULL""").collect()}


def restore_version(table: str):
    """Manifest restore_version for a FROZEN v3 / v4-DRAFT table, or None (then only the current-table source is tried)."""
    return RESTORE_VERSION.get(table.split(".")[-1])


def fingerprint(df):
    """(n, v1 NULL-skip, v2 NULL-token) per freeze_pin_fingerprint_gate. BUG-QA-098: gating on v1 alone misses NULL shifts."""
    from pyspark.sql import functions as F
    h1 = F.sha2(F.concat_ws("|", *[F.col(f"`{c}`").cast("string") for c in df.columns]), 256)
    h2 = F.sha2(F.concat_ws("|", *[F.coalesce(F.col(f"`{c}`").cast("string"), F.lit("<NULL>")) for c in df.columns]), 256)
    r = df.select(F.count("*").alias("n"),
                  F.sum(F.conv(F.substring(h1, 1, 15), 16, 10).cast("decimal(38,0)")).alias("h1"),
                  F.sum(F.conv(F.substring(h2, 1, 15), 16, 10).cast("decimal(38,0)")).alias("h2")).first()
    return (int(r.n), int(r.h1 or 0), int(r.h2 or 0))


def v3src(table: str, pin: int) -> str:
    """Return a temp view holding the data of table@pin, gated on PIN_FP. Fails closed.
    Tries 1) current table (+ PIN_FILTER), 2) VERSION AS OF demo_freeze_manifest.restore_version (+ PIN_FILTER; P67,
    BUG-QA-095), never the delta_version pin, which may be past the 7-day retention. The first source whose
    fingerprint equals PIN_FP wins; if none matches, raise AssertionError and nothing downstream runs."""
    key = (table.split(".")[-1], int(pin))
    if key in _V3SRC:
        return _V3SRC[key]
    assert key in PIN_FP, f"no recorded fingerprint for {key}; refusing an ungated read"
    full = table if "." in table else f"{S}.{table}"
    filt = PIN_FILTER.get(key)
    tried = []
    rv = restore_version(key[0])
    sources = [("current", f"SELECT * FROM {full}" + (f" WHERE {filt}" if filt else ""))]
    if rv is not None:
        sources.append((f"restore_version@{rv}", f"SELECT * FROM {full} VERSION AS OF {rv}" + (f" WHERE {filt}" if filt else "")))
    for mode, q in sources:
        try:
            got = fingerprint(spark.sql(q))
        except Exception as e:  # e.g. DELTA_UNSUPPORTED_TIME_TRAVEL_BEYOND_DELETED_FILE_RETENTION_DURATION
            tried.append((mode, type(e).__name__))
            continue
        if got == PIN_FP[key]:
            view = f"fp_{key[0]}_{key[1]}"
            spark.sql(f"CREATE OR REPLACE TEMP VIEW {view} AS {q}")
            gate(f"FP {key[0]}@{key[1]} via {mode}", True, got)
            _V3SRC[key] = view
            return view
        tried.append((mode, got))
    gate(f"FP {key[0]}@{key[1]}", False, {"want": PIN_FP[key], "tried": tried})
    raise AssertionError(f"fingerprint gate failed closed for {key}: {tried}")

# Fixed build constants, so a rerun reproduces the live candidates exactly
B2_BUILD_TS = "2026-10-25 21:09:47.991845"   # created_at of the 92 CDT-BBR-* conduits
HSI_BUILD_TS = "2026-10-25 13:14:35.27262"   # computed_at of every HSI candidate row
HSI_BATCH_ID = "v4cand-bugqa065-064-20261025"
MAX_SEG_KM = 25.0      # BBS segment cap (B1)
ARC_STEPS = 20000      # arc-length sampling steps along each planar route chord (BUG-QA-075)
DENSIFY_KM = 5.0       # great-circle densification step for HSI span lines (BUG-QA-065)
R_KM = 6371.0088       # mean Earth radius used by every haversine in this notebook

GATES = []


def assert_v4_target(name: str) -> None:
    """Runtime companion to tests/test_v4_frozen_write_guard.py."""
    assert name.startswith(S + ".") and name.endswith("_v4_candidate"), f"refusing non-v4 target {name}"


for _t in (SPAN_V4, CONDUIT_V4, HSI_V4, ALARM_V4, PERF_V4, MV_MRC_V4, MV_HPC_V4):
    assert_v4_target(_t)


def gate(name: str, ok: bool, detail) -> None:
    GATES.append({"gate": name, "pass": bool(ok), "detail": str(detail)})
    print(("PASS " if ok else "FAIL ") + name + ": " + str(detail))


def diff_vs_live(view: str, target: str, key: str, label: str) -> None:
    """Dry-run gate: recomputed temp view vs live table, full outer join on key and xxhash64 over all columns."""
    cols = spark.table(target).columns
    assert spark.table(view).columns == cols, f"{label}: column order/name drift vs {target}"
    h = lambda a: "xxhash64(" + ", ".join(f"{a}.`{c}`" for c in cols) + ")"
    r = spark.sql(f"""
      SELECT count(r.{key}) AS n_rec, count(t.{key}) AS n_live,
             coalesce(sum(int(r.{key} IS NULL)), 0) AS only_live,
             coalesce(sum(int(t.{key} IS NULL)), 0) AS only_rec,
             coalesce(sum(int(r.{key} IS NOT NULL AND t.{key} IS NOT NULL AND {h('r')} <> {h('t')})), 0) AS val_diff
      FROM {view} r FULL OUTER JOIN {target} t ON r.{key} = t.{key}""").collect()[0].asDict()
    ok = r["n_rec"] == r["n_live"] and r["only_live"] == 0 and r["only_rec"] == 0 and r["val_diff"] == 0
    gate(f"{label} recompute == live", ok, r)


def hav(lon1, lat1, lon2, lat2):
    p1, p2 = np.radians(lat1), np.radians(lat2)
    a = np.sin((p2 - p1) / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(np.radians(lon2 - lon1) / 2) ** 2
    return 2 * R_KM * np.arcsin(np.sqrt(a))


def parse_wkt(wkt: str):
    return [(float(x), float(y)) for x, y in re.findall(r"(-?[\d.]+) (-?[\d.]+)", wkt)]


def segment_route(a, z, max_km=MAX_SEG_KM, steps=ARC_STEPS):
    """BUG-QA-075: split the planar lon/lat chord a->z into n = ceil(arc_km / max_km) pieces of equal haversine arc length."""
    t = np.linspace(0, 1, steps + 1)
    lon, lat = a[0] + t * (z[0] - a[0]), a[1] + t * (z[1] - a[1])
    cum = np.concatenate([[0], np.cumsum(hav(lon[:-1], lat[:-1], lon[1:], lat[1:]))])
    n = max(1, math.ceil(cum[-1] / max_km))
    tk = np.interp(np.arange(n + 1) * cum[-1] / n, cum, t)
    tk[0], tk[-1] = 0.0, 1.0
    return n, [(a[0] + x * (z[0] - a[0]), a[1] + x * (z[1] - a[1])) for x in tk]


def slerp_densify_r6(coords, step_km=DENSIFY_KM):
    """BUG-QA-065: densify each vertex pair along the great circle (n = ceil(km / step)), coordinates rounded to 6 dp."""
    out = [coords[0]]
    for (lon1, lat1), (lon2, lat2) in zip(coords[:-1], coords[1:]):
        n = max(1, math.ceil(float(hav(lon1, lat1, lon2, lat2)) / step_km))
        if n > 1:
            v1 = np.array([math.cos(math.radians(lat1)) * math.cos(math.radians(lon1)),
                           math.cos(math.radians(lat1)) * math.sin(math.radians(lon1)), math.sin(math.radians(lat1))])
            v2 = np.array([math.cos(math.radians(lat2)) * math.cos(math.radians(lon2)),
                           math.cos(math.radians(lat2)) * math.sin(math.radians(lon2)), math.sin(math.radians(lat2))])
            om = math.acos(max(-1.0, min(1.0, float(v1 @ v2))))
            for k in range(1, n):
                f = k / n
                v = (math.sin((1 - f) * om) * v1 + math.sin(f * om) * v2) / math.sin(om)
                out.append((math.degrees(math.atan2(v[1], v[0])), math.degrees(math.asin(v[2]))))
        out.append((lon2, lat2))
    return [(round(x, 6), round(y, 6)) for x, y in out]


print(f"write_enabled={WRITE_ENABLED}  (dry run diffs only)" if not WRITE_ENABLED else "write_enabled=True: v4 candidate objects WILL be overwritten")

In [0]:
# 10,000 SPAN-* rows unchanged from span@20; 92 BB-* routes replaced by BBS-<route>-NNN segments on CDT-BBR-<route>
span_cols = spark.table(f"{S}.physical_resource_span").columns
bb_rows = [r.asDict() for r in spark.sql(
    f"SELECT * FROM {v3src('physical_resource_span', PIN_SPAN)} WHERE span_id LIKE 'BB-%'").collect()]

bbs = []
for r in bb_rows:
    v = parse_wkt(r["geom_wkt"])
    n, pts = segment_route(v[0], v[-1])
    segs = [(k, *pts[k - 1], *pts[k]) for k in range(1, n + 1)]
    seg_km = [float(hav(x1, y1, x2, y2)) for _, x1, y1, x2, y2 in segs]
    route_km = round(sum(seg_km), 3)
    rid = r["span_id"].split("-")[1]
    for (k, x1, y1, x2, y2), d in zip(segs, seg_km):
        bbs.append(dict(
            span_id=f"BBS-{rid}-{k:03d}", span_name=f"{r['span_name']} seg {k}/{n}", conduit_id=f"CDT-BBR-{rid}",
            site_a_id=r["site_a_id"], site_z_id=r["site_z_id"], fiber_type=r["fiber_type"], fiber_count=r["fiber_count"],
            wavelength_count=r["wavelength_count"], length_km=round(d, 3),
            geom_wkt=f"LINESTRING ({x1:.10f} {y1:.10f}, {x2:.10f} {y2:.10f})",
            install_year=r["install_year"], is_active=r["is_active"], created_at=r["created_at"],
            midpoint_lat=round((y1 + y2) / 2, 6), midpoint_lon=round((x1 + x2) / 2, 6), origin="DERIVED",
            parent_route_id=r["span_id"], segment_seq=k, segment_count=n,
            route_length_km_v3=r["length_km"], route_geodesic_km=route_km))

SPAN_V4_SCHEMA = [("span_id", "string"), ("span_name", "string"), ("conduit_id", "string"), ("site_a_id", "string"),
                  ("site_z_id", "string"), ("fiber_type", "string"), ("fiber_count", "int"), ("wavelength_count", "int"),
                  ("length_km", "double"), ("geom_wkt", "string"), ("install_year", "int"), ("is_active", "boolean"),
                  ("created_at", "timestamp"), ("midpoint_lat", "double"), ("midpoint_lon", "double"), ("origin", "string"),
                  ("parent_route_id", "string"), ("segment_seq", "int"), ("segment_count", "int"),
                  ("route_length_km_v3", "double"), ("route_geodesic_km", "double")]
bbs_df = spark.createDataFrame(pd.DataFrame(bbs)).selectExpr(*[f"CAST({c} AS {t}) AS {c}" for c, t in SPAN_V4_SCHEMA])
spans_df = spark.sql(f"""
  SELECT {', '.join(span_cols)}, CAST(NULL AS STRING) AS parent_route_id, CAST(NULL AS INT) AS segment_seq,
         CAST(NULL AS INT) AS segment_count, CAST(NULL AS DOUBLE) AS route_length_km_v3, CAST(NULL AS DOUBLE) AS route_geodesic_km
  FROM {v3src('physical_resource_span', PIN_SPAN)} WHERE span_id NOT LIKE 'BB-%'""")
spans_df.select(*[c for c, _ in SPAN_V4_SCHEMA]).unionByName(bbs_df).createOrReplaceTempView("v4_span")

g = spark.sql("""SELECT count(*) n, count_if(span_id LIKE 'BBS-%') n_bbs, count_if(span_id LIKE 'BB-%' AND span_id NOT LIKE 'BBS-%') n_bb,
  max(CASE WHEN span_id LIKE 'BBS-%' THEN length_km END) max_bbs_km, count(DISTINCT parent_route_id) routes FROM v4_span""").collect()[0]
gate("B1 BBS segments <= 25 km, 92 routes, 0 BB-* left", g.max_bbs_km <= MAX_SEG_KM and g.routes == 92 and g.n_bb == 0, g.asDict())

if WRITE_ENABLED:
    assert_v4_target(SPAN_V4)
    spark.sql(f"CREATE TABLE IF NOT EXISTS {SPAN_V4} AS SELECT * FROM v4_span WHERE 1 = 0")
    spark.sql(f"INSERT OVERWRITE {SPAN_V4} SELECT * FROM v4_span")
diff_vs_live("v4_span", SPAN_V4, "span_id", "B1 span_v4")

In [0]:
# A1: length_km = haversine sum over geom_wkt vertices (3 dp); P35/B2 coverage from v4_span; 92 CDT-BBR-* route conduits
HAV = lambda lat1, lon1, lat2, lon2: (
    f"2*{R_KM}*ASIN(SQRT(POWER(SIN(RADIANS({lat2}-{lat1})/2),2)+COS(RADIANS({lat1}))*COS(RADIANS({lat2}))*POWER(SIN(RADIANS({lon2}-{lon1})/2),2)))")
conduit_sql = f"""
WITH v3 AS (SELECT * FROM {v3src('physical_resource_conduit', PIN_CONDUIT)}),
bb AS (SELECT * FROM {v3src('physical_resource_span', PIN_SPAN)} WHERE span_id LIKE 'BB-%'),
lines AS (SELECT conduit_id AS line_id, geom_wkt FROM v3 UNION ALL SELECT span_id, geom_wkt FROM bb),
pts AS (
  SELECT l.line_id, p.pos, CAST(split(trim(p.xy), ' ')[0] AS DOUBLE) AS lon, CAST(split(trim(p.xy), ' ')[1] AS DOUBLE) AS lat
  FROM lines l LATERAL VIEW posexplode(split(regexp_extract(l.geom_wkt, '\\\\((.*)\\\\)', 1), ',')) p AS pos, xy),
hv AS (
  SELECT line_id, pos, lon, lat,
         LAG(lon) OVER (PARTITION BY line_id ORDER BY pos) AS plon, LAG(lat) OVER (PARTITION BY line_id ORDER BY pos) AS plat
  FROM pts),
geo AS (
  SELECT line_id, ROUND(SUM(CASE WHEN plon IS NULL THEN 0 ELSE {HAV('plat', 'plon', 'lat', 'lon')} END), 3) AS geo_km,
         MIN_BY(struct(lon, lat), pos) AS f, MAX_BY(struct(lon, lat), pos) AS l
  FROM hv GROUP BY line_id),
geo2 AS (SELECT line_id, geo_km, ROUND({HAV('f.lat', 'f.lon', 'l.lat', 'l.lon')}, 3) AS chord_km FROM geo),
real_seg AS (SELECT conduit_id, ROUND(SUM(length_km), 3) AS km FROM v4_span WHERE span_id LIKE 'SPAN-%' GROUP BY 1),
bbs AS (SELECT conduit_id, ROUND(SUM(length_km), 3) AS km FROM v4_span WHERE span_id LIKE 'BBS-%' GROUP BY 1),
base AS (
  SELECT c.conduit_id, c.conduit_name, c.corridor, c.owner, c.conduit_type, c.install_year, c.site_a_id, c.site_z_id, c.geom_wkt,
         g.geo_km AS length_km, c.is_active, c.created_at, c.origin, c.length_km AS length_km_v3, g.chord_km,
         ROUND(g.geo_km - c.length_km, 3) AS length_delta_km
  FROM v3 c JOIN geo2 g ON g.line_id = c.conduit_id)
SELECT b.*,
  CASE WHEN ABS(b.length_delta_km) <= 0.0005 THEN 'UNCHANGED'
       WHEN ABS(b.length_delta_km) >= 0.01 * b.length_km THEN 'CORRECTED_GE_1PCT'
       ELSE 'CORRECTED_LT_1PCT' END AS length_change_class,
  'geodesic_sum_of_vertex_segments' AS length_method,
  CASE WHEN r.km IS NOT NULL THEN 'REAL_SEGMENTED' ELSE 'INVENTORY_ONLY' END AS span_coverage_class,
  COALESCE(r.km, 0.0) AS segmented_km,
  CASE WHEN r.km IS NULL THEN 0.0 ELSE LEAST(100.00, ROUND(100 * r.km / b.length_km, 2)) END AS coverage_pct
FROM base b LEFT JOIN real_seg r USING (conduit_id)
UNION ALL
SELECT concat('CDT-BBR-', split(s.span_id, '-')[1]),
  concat('Backbone route conduit ', s.site_a_id, '-', s.site_z_id, ' (', s.span_id, ')'),
  concat('Backbone ', s.site_a_id, '-', s.site_z_id), p.owner, p.conduit_type, s.install_year, s.site_a_id, s.site_z_id, s.geom_wkt,
  g.geo_km, s.is_active, TIMESTAMP'{B2_BUILD_TS}', 'DERIVED', CAST(NULL AS DOUBLE), g.chord_km, CAST(NULL AS DOUBLE), 'NEW_V4',
  'geodesic_sum_of_vertex_segments', 'BB_SEGMENTED', k.km, LEAST(100.00, ROUND(100 * k.km / g.geo_km, 2))
FROM bb s
JOIN v3 p ON p.conduit_id = s.conduit_id
JOIN geo2 g ON g.line_id = s.span_id
JOIN bbs k ON k.conduit_id = concat('CDT-BBR-', split(s.span_id, '-')[1])
"""
spark.sql(conduit_sql).createOrReplaceTempView("v4_conduit")

g = spark.sql("""SELECT count(*) n, count_if(span_coverage_class='REAL_SEGMENTED') real_seg, count_if(span_coverage_class='BB_SEGMENTED') bb_seg,
  count_if(span_coverage_class='INVENTORY_ONLY') inv_only FROM v4_conduit""").collect()[0]
gate("P35/B2 coverage classes 55/92/1914", (g.real_seg, g.bb_seg, g.inv_only) == (55, 92, 1914), g.asDict())
orph = spark.sql("SELECT count(*) c FROM v4_span s LEFT ANTI JOIN v4_conduit c USING (conduit_id)").collect()[0].c
gate("B2 span_v4 -> conduit_v4 FK orphans = 0", orph == 0, orph)

if WRITE_ENABLED:
    assert_v4_target(CONDUIT_V4)
    spark.sql(f"CREATE TABLE IF NOT EXISTS {CONDUIT_V4} AS SELECT * FROM v4_conduit WHERE 1 = 0")
    spark.sql(f"INSERT OVERWRITE {CONDUIT_V4} SELECT * FROM v4_conduit")
diff_vs_live("v4_conduit", CONDUIT_V4, "conduit_id", "A1/B2 conduit_v4")

In [0]:
# Rows = every v3 pair (@22, re-measured) + new in-radius pairs found by the line test. Measured on span@11 (v3 BB-* ids; BBS rebuild is BUG-QA-073).
# Distance: span line densified along great circles (5 km, 6 dp) -> closest point in EPSG:5070 -> st_distancesphere from hazard point.
sp11 = spark.sql(f"SELECT span_id, geom_wkt FROM {v3src('physical_resource_span', PIN_SPAN_HSI)}").collect()
dense = [(r.span_id, "LINESTRING (" + ", ".join(f"{x!r} {y!r}" for x, y in slerp_densify_r6(parse_wkt(r.geom_wkt))) + ")") for r in sp11]
spark.createDataFrame(dense, "span_id string, geom_dense_wkt string").createOrReplaceTempView("v4_hsi_span_dense")

spark.sql(f"""
  SELECT s.span_id, s.conduit_id, s.midpoint_lat, s.midpoint_lon, s.geom_wkt,
         st_transform(st_geomfromtext(d.geom_dense_wkt, 4326), 5070) AS g5070,
         st_xmin(st_geomfromtext(d.geom_dense_wkt)) AS xmin, st_xmax(st_geomfromtext(d.geom_dense_wkt)) AS xmax,
         st_ymin(st_geomfromtext(d.geom_dense_wkt)) AS ymin, st_ymax(st_geomfromtext(d.geom_dense_wkt)) AS ymax
  FROM {v3src('physical_resource_span', PIN_SPAN_HSI)} s JOIN v4_hsi_span_dense d USING (span_id)""").createOrReplaceTempView("v4_hsi_sp")
spark.sql(f"""
  SELECT hazard_id, hazard_type, severity, latitude, longitude, radius_km, event_start, event_end, source,
         st_transform(st_point(longitude, latitude, 4326), 5070) AS p5070,
         radius_km / 111.0 + 0.01 AS dlat, radius_km / (111.0 * greatest(cos(radians(latitude)), 0.2)) + 0.01 AS dlon
  FROM {v3src('geographic_hazard_event', PIN_HAZARD)}""").createOrReplaceTempView("v4_hsi_hz")

# P90 / DP standard (P90-DP-HSI-DEDUP-RULE-ID-FORMULA-REVIEW): an intersection_id is formula-valid if it matches the CURRENT
# formula SHA2(hazard_source||hazard_source_id||span_id,256) OR the LEGACY formula (batch-20261021-171107)
# SHA2(CONCAT_WS('|', staging_label, hazard_source_id, span_id),256). No column stores staging_label, so derive it from hazard_type.
# Dedup per (span_id, hazard_id): formula-valid row first, tiebreak min(intersection_id). Shared by cell 5 and cell 5b.
HSI_LEGACY_LABEL = ("CASE hazard_type WHEN 'earthquake' THEN 'usgs_earthquake' WHEN 'severe_weather' THEN 'noaa_weather' "
                    "WHEN 'wildfire' THEN 'nifc_wildfire' WHEN 'flood' THEN 'usgs_flood' WHEN 'dig_ticket' THEN '811_dig_ticket' END")
HSI_ID_VALID = ("(intersection_id = sha2(concat(hazard_source, hazard_source_id, span_id), 256) OR "
                f"intersection_id = sha2(concat_ws('|', {HSI_LEGACY_LABEL}, hazard_source_id, span_id), 256))")
HSI_DEDUP_ORDER = f"CASE WHEN {HSI_ID_VALID} THEN 0 ELSE 1 END, intersection_id"

hsi_sql = f"""
WITH v3_raw AS (SELECT * FROM {v3src('hazard_span_intersection', PIN_HSI)}),
v3 AS (  -- BUG-QA-111: v3@22 has 6 duplicate (span_id, hazard_id) pairs (batch-20261024-bugqa061-063; same hazard_source_id and
         -- distance, different intersection_id). Keep the formula-valid row (current OR legacy formula, HSI_DEDUP_ORDER);
         -- tiebreak min(intersection_id). 173,323 -> 173,317 v3 rows; v4 200,089 -> 200,083.
  SELECT * EXCEPT (_rn) FROM (
    SELECT *, row_number() OVER (PARTITION BY span_id, hazard_id ORDER BY {HSI_DEDUP_ORDER}) AS _rn
    FROM v3_raw) WHERE _rn = 1),
grid_pairs AS (  -- 1-degree grid buckets, then the exact bbox test (bbox = radius + 0.01 deg margin)
  SELECT DISTINCT h.hazard_id, s.span_id FROM
    (SELECT hazard_id, gx, gy FROM v4_hsi_hz
       LATERAL VIEW explode(sequence(CAST(floor(longitude - dlon) AS INT), CAST(floor(longitude + dlon) AS INT))) a AS gx
       LATERAL VIEW explode(sequence(CAST(floor(latitude - dlat) AS INT), CAST(floor(latitude + dlat) AS INT))) b AS gy) h
    JOIN (SELECT span_id, gx, gy FROM v4_hsi_sp
       LATERAL VIEW explode(sequence(CAST(floor(xmin) AS INT), CAST(floor(xmax) AS INT))) a AS gx
       LATERAL VIEW explode(sequence(CAST(floor(ymin) AS INT), CAST(floor(ymax) AS INT))) b AS gy) s
    USING (gx, gy)),
cand AS (
  SELECT h.hazard_id, s.span_id, h.radius_km,
         st_distancesphere(st_point(h.longitude, h.latitude, 4326), st_transform(st_closestpoint(s.g5070, h.p5070), 4326)) / 1000 AS line_km
  FROM grid_pairs k JOIN v4_hsi_hz h USING (hazard_id) JOIN v4_hsi_sp s USING (span_id)
  WHERE s.xmax >= h.longitude - h.dlon AND s.xmin <= h.longitude + h.dlon
    AND s.ymax >= h.latitude - h.dlat AND s.ymin <= h.latitude + h.dlat),
newp AS (
  SELECT c.hazard_id, c.span_id FROM cand c
  LEFT ANTI JOIN (SELECT DISTINCT hazard_id, span_id FROM v3) x USING (hazard_id, span_id)
  WHERE c.line_km <= c.radius_km),
span_svc AS (SELECT span_id, first(affected_service_ids) AS affected_service_ids, first(affected_service_cnt) AS affected_service_cnt
             FROM v3 GROUP BY span_id),
base AS (
  SELECT v3.intersection_id, v3.hazard_source, v3.hazard_source_id, v3.hazard_type, v3.hazard_severity,
         v3.hazard_id, v3.span_id, v3.affected_service_ids, v3.affected_service_cnt, v3.is_within_radius AS v3_is_within_radius,
         v3.distance_km AS v3_distance_km, v3.batch_id AS v3_batch_id, 'V3_PAIR' AS pair_origin
  FROM v3
  UNION ALL
  SELECT sha2(concat(h.source, h.hazard_id, n.span_id), 256), h.source, h.hazard_id, h.hazard_type, h.severity,
         n.hazard_id, n.span_id, s.affected_service_ids, s.affected_service_cnt,
         CAST(NULL AS BOOLEAN), CAST(NULL AS DOUBLE), CAST(NULL AS STRING), 'NEW_LINE_PAIR'
  FROM newp n JOIN v4_hsi_hz h USING (hazard_id) LEFT JOIN span_svc s USING (span_id)),
m AS (
  SELECT b.*, h.latitude AS hazard_lat, h.longitude AS hazard_lon, h.radius_km AS hazard_radius_km,
         h.event_start AS hazard_start, h.event_end AS hazard_end,
         sp.midpoint_lat AS span_midpoint_lat, sp.midpoint_lon AS span_midpoint_lon,
         st_transform(st_closestpoint(sp.g5070, h.p5070), 4326) AS np
  FROM base b JOIN v4_hsi_hz h USING (hazard_id) JOIN v4_hsi_sp sp USING (span_id)),
d AS (
  SELECT *, ROUND(st_distancesphere(st_point(hazard_lon, hazard_lat, 4326), np) / 1000, 4) AS distance_km,
         ROUND(st_distancesphere(st_point(hazard_lon, hazard_lat), st_point(span_midpoint_lon, span_midpoint_lat)) / 1000, 4) AS distance_to_midpoint_km
  FROM m)
SELECT intersection_id, hazard_source, hazard_source_id, hazard_type, hazard_severity, hazard_lat, hazard_lon, hazard_radius_km,
  hazard_start, hazard_end, span_id, span_midpoint_lat, span_midpoint_lon, distance_km, distance_to_midpoint_km,
  ROUND(st_y(np), 6) AS nearest_point_lat, ROUND(st_x(np), 6) AS nearest_point_lon,
  distance_km <= hazard_radius_km AS is_within_radius,
  CASE WHEN distance_km < 1 THEN 'direct_impact' WHEN distance_km < 5 THEN 'near_field' ELSE 'buffer_zone' END AS proximity_band,
  affected_service_ids, affected_service_cnt, TIMESTAMP'{HSI_BUILD_TS}' AS computed_at, '{HSI_BATCH_ID}' AS batch_id,
  hazard_id, 'DERIVED' AS origin, pair_origin, v3_is_within_radius, v3_distance_km, v3_batch_id,
  CASE WHEN pair_origin = 'NEW_LINE_PAIR' THEN 'GAINED_NEW_PAIR'
       WHEN v3_is_within_radius AND distance_km <= hazard_radius_km THEN 'UNCHANGED_IN'
       WHEN NOT v3_is_within_radius AND distance_km > hazard_radius_km THEN 'UNCHANGED_OUT'
       WHEN NOT v3_is_within_radius THEN 'GAINED_FLAG_FLIP'
       ELSE 'LOST' END AS change_vs_v3,
  'geodesic_to_linestring' AS distance_method
FROM d
"""
spark.sql(hsi_sql).createOrReplaceTempView("v4_hsi")

# BUG-QA-111 uniqueness gate (also the P43 PK precondition): one row per (span_id, hazard_id) and per intersection_id.
_g111 = spark.sql("""SELECT count(*) AS n, count(*) - count(DISTINCT span_id, hazard_id) AS dup_pairs,
  count(*) - count(DISTINCT intersection_id) AS dup_ids FROM v4_hsi""").first()
assert _g111.dup_pairs == 0 and _g111.dup_ids == 0, f"FAIL BUG-QA-111 uniqueness gate: {_g111.asDict()}"
print("PASS BUG-QA-111 (span_id, hazard_id) unique:", _g111.asDict())

# BUG-QA-126/127 time gate: hazard_start/hazard_end are denormalized from geographic_hazard_event (v4_hsi_hz, ghe@PIN_HAZARD) by
# hazard_id, never carried from v3. v3 HSI has 4,683 rows (batch-20261021-171107) with hazard_end truncated to start-date midnight
# and 18,427 NIFC rows (batch-20261024-bugqa061-063) holding pre-P17 (-5 y) times. Gate: 0 end<start, 0 mismatches vs ghe.
_g126 = spark.sql("""SELECT sum(int(h.hazard_end < h.hazard_start)) AS end_lt_start,
  sum(int(NOT (h.hazard_start <=> z.event_start AND h.hazard_end <=> z.event_end))) AS time_mm,
  sum(int(h.hazard_source = 'nifc' AND NOT (h.hazard_start <=> z.event_start AND h.hazard_end <=> z.event_end))) AS nifc_mm,
  sum(int(z.hazard_id IS NULL)) AS no_ghe
  FROM v4_hsi h LEFT JOIN v4_hsi_hz z USING (hazard_id)""").first()
assert all((v or 0) == 0 for v in _g126.asDict().values()), f"FAIL BUG-QA-126/127 time gate: {_g126.asDict()}"
print("PASS BUG-QA-126/127 hazard times == ghe:", _g126.asDict())

if WRITE_ENABLED:
    assert_v4_target(HSI_V4)
    spark.sql(f"CREATE TABLE IF NOT EXISTS {HSI_V4} AS SELECT * FROM v4_hsi WHERE 1 = 0")
    spark.sql(f"INSERT OVERWRITE {HSI_V4} SELECT * FROM v4_hsi")
diff_vs_live("v4_hsi", HSI_V4, "intersection_id", "HSI_v4")  # about 7 min on serverless

In [0]:
# BUG-QA-073 + BUG-DP-071 (@data-planner). Supersedes cell 5 as the HSI v4 writer once approved.
# SPAN-* rows: carried from HSI_v4@33, 6 duplicate (span_id, hazard_id) pairs deduped with cell 5's HSI_DEDUP_ORDER
#   (P90-DE-ALIGN-5B-DEDUP-RULE: formula-valid id first, current OR legacy formula; tiebreak min(intersection_id)).
# BBS-* rows: re-derived on physical_resource_span_v4_candidate BBS segments vs all hazards @34 (bbox prefilter + every
#   segment of each v3 BB route pair), same distance method as cell 5 (5 km slerp densify, EPSG:5070 closest point, st_distancesphere).
# Labels: change_vs_v3 at parent_route grain vs v3 BB pairs (@22). DEC-PRI-BBS-NEW-OUT-LABEL: new out-of-radius route pairs are DROPPED.
# Out-of-radius v3 route pairs keep exactly one row (nearest segment). affected_service_* inherited from the v3 parent BB route.
# Write is double-gated: write_enabled=true AND qa073_write_approved=true (CEO escalation #34). Default = dry run + gates A-G.
dbutils.widgets.dropdown("qa073_write_approved", "false", ["false", "true"], "BUG-QA-073 HSI overwrite approved")
QA073_WRITE = WRITE_ENABLED and dbutils.widgets.get("qa073_write_approved") == "true"
PIN_HSI_V4_SRC = 33
QA073_TS = "2026-10-27 18:00:00"
QA073_BATCH = "v4cand-bugqa073-bbs-20261027"

bbs_rows = spark.sql(f"SELECT span_id, geom_wkt FROM {SPAN_V4} WHERE span_id LIKE 'BBS-%'").collect()
spark.createDataFrame([(r.span_id, "LINESTRING (" + ", ".join(f"{x!r} {y!r}" for x, y in slerp_densify_r6(parse_wkt(r.geom_wkt))) + ")")
                       for r in bbs_rows], "span_id string, geom_dense_wkt string").createOrReplaceTempView("q73_dense")
spark.sql(f"""SELECT s.span_id, s.parent_route_id, s.midpoint_lat, s.midpoint_lon,
  st_transform(st_geomfromtext(d.geom_dense_wkt, 4326), 5070) AS g5070,
  st_xmin(st_geomfromtext(d.geom_dense_wkt)) AS xmin, st_xmax(st_geomfromtext(d.geom_dense_wkt)) AS xmax,
  st_ymin(st_geomfromtext(d.geom_dense_wkt)) AS ymin, st_ymax(st_geomfromtext(d.geom_dense_wkt)) AS ymax
  FROM {SPAN_V4} s JOIN q73_dense d USING (span_id)""").cache().createOrReplaceTempView("q73_sp")
spark.table("v4_hsi_hz").cache().createOrReplaceTempView("q73_hz")  # hazards @PIN_HAZARD, built in cell 5
spark.sql(f"""SELECT span_id AS parent_route_id, hazard_id, max(is_within_radius) AS v3_in, min(distance_km) AS v3_dist, first(batch_id) AS v3_batch
  FROM {v3src('hazard_span_intersection', PIN_HSI)} WHERE span_id LIKE 'BB-%' GROUP BY 1, 2""").cache().createOrReplaceTempView("q73_v3")
spark.sql(f"""SELECT span_id AS parent_route_id, first(affected_service_ids) AS svc_ids, first(affected_service_cnt) AS svc_cnt
  FROM {v3src('hazard_span_intersection', PIN_HSI)} WHERE span_id LIKE 'BB-%' GROUP BY 1""").createOrReplaceTempView("q73_route_svc")

spark.sql("""
WITH grid AS (SELECT DISTINCT h.hazard_id, s.span_id FROM
  (SELECT hazard_id, gx, gy FROM q73_hz LATERAL VIEW explode(sequence(CAST(floor(longitude - dlon) AS INT), CAST(floor(longitude + dlon) AS INT))) a AS gx
     LATERAL VIEW explode(sequence(CAST(floor(latitude - dlat) AS INT), CAST(floor(latitude + dlat) AS INT))) b AS gy) h
  JOIN (SELECT span_id, gx, gy FROM q73_sp LATERAL VIEW explode(sequence(CAST(floor(xmin) AS INT), CAST(floor(xmax) AS INT))) a AS gx
     LATERAL VIEW explode(sequence(CAST(floor(ymin) AS INT), CAST(floor(ymax) AS INT))) b AS gy) s USING (gx, gy)),
bbox AS (SELECT k.hazard_id, k.span_id FROM grid k JOIN q73_hz h USING (hazard_id) JOIN q73_sp s USING (span_id)
  WHERE s.xmax >= h.longitude - h.dlon AND s.xmin <= h.longitude + h.dlon AND s.ymax >= h.latitude - h.dlat AND s.ymin <= h.latitude + h.dlat),
v3seg AS (SELECT v.hazard_id, s.span_id FROM q73_v3 v JOIN q73_sp s USING (parent_route_id)),
pairs AS (SELECT * FROM bbox UNION SELECT * FROM v3seg),
c0 AS (SELECT p.hazard_id, p.span_id, s.parent_route_id, st_transform(st_closestpoint(s.g5070, h.p5070), 4326) AS np
       FROM pairs p JOIN q73_hz h USING (hazard_id) JOIN q73_sp s USING (span_id))
SELECT c0.*, h.radius_km, ROUND(st_distancesphere(st_point(h.longitude, h.latitude, 4326), c0.np) / 1000, 4) AS distance_km
FROM c0 JOIN q73_hz h USING (hazard_id)""").cache().createOrReplaceTempView("q73_cand")

spark.sql(f"""
WITH c AS (SELECT *, distance_km <= radius_km AS seg_in FROM q73_cand),
route AS (SELECT parent_route_id, hazard_id, max(seg_in) AS v4_in FROM c GROUP BY 1, 2),
lab AS (SELECT r.parent_route_id, r.hazard_id, r.v4_in, v.v3_in, v.v3_dist, v.v3_batch,
  CASE WHEN v.hazard_id IS NULL AND r.v4_in THEN 'GAINED_NEW_PAIR'
       WHEN v.hazard_id IS NULL THEN 'DROP_NEW_OUT'
       WHEN v.v3_in AND r.v4_in THEN 'UNCHANGED_IN'
       WHEN v.v3_in THEN 'LOST'
       WHEN r.v4_in THEN 'GAINED_FLAG_FLIP'
       ELSE 'UNCHANGED_OUT' END AS change_vs_v3
  FROM route r LEFT JOIN q73_v3 v USING (parent_route_id, hazard_id)),
keep AS (SELECT c.*, l.change_vs_v3, l.v3_in, l.v3_dist, l.v3_batch, l.v4_in,
  row_number() OVER (PARTITION BY c.parent_route_id, c.hazard_id ORDER BY c.distance_km, c.span_id) AS rn
  FROM c JOIN lab l USING (parent_route_id, hazard_id) WHERE l.change_vs_v3 <> 'DROP_NEW_OUT')
SELECT sha2(concat(h.source, k.hazard_id, k.span_id), 256) AS intersection_id, h.source AS hazard_source, k.hazard_id AS hazard_source_id,
  h.hazard_type, h.severity AS hazard_severity, h.latitude AS hazard_lat, h.longitude AS hazard_lon, h.radius_km AS hazard_radius_km,
  h.event_start AS hazard_start, h.event_end AS hazard_end, k.span_id, sp.midpoint_lat AS span_midpoint_lat, sp.midpoint_lon AS span_midpoint_lon,
  k.distance_km, ROUND(st_distancesphere(st_point(h.longitude, h.latitude), st_point(sp.midpoint_lon, sp.midpoint_lat)) / 1000, 4) AS distance_to_midpoint_km,
  ROUND(st_y(k.np), 6) AS nearest_point_lat, ROUND(st_x(k.np), 6) AS nearest_point_lon, k.seg_in AS is_within_radius,
  CASE WHEN k.distance_km < 1 THEN 'direct_impact' WHEN k.distance_km < 5 THEN 'near_field' ELSE 'buffer_zone' END AS proximity_band,
  rs.svc_ids AS affected_service_ids, rs.svc_cnt AS affected_service_cnt, TIMESTAMP'{QA073_TS}' AS computed_at, '{QA073_BATCH}' AS batch_id,
  k.hazard_id, 'DERIVED' AS origin, CASE WHEN k.v3_in IS NULL THEN 'NEW_LINE_PAIR' ELSE 'V3_PAIR' END AS pair_origin,
  k.v3_in AS v3_is_within_radius, k.v3_dist AS v3_distance_km, k.v3_batch AS v3_batch_id, k.change_vs_v3, 'geodesic_to_linestring' AS distance_method
FROM keep k JOIN q73_hz h USING (hazard_id) JOIN q73_sp sp USING (span_id) JOIN q73_route_svc rs ON rs.parent_route_id = k.parent_route_id
WHERE k.seg_in OR (NOT k.v4_in AND k.rn = 1)""").createOrReplaceTempView("q73_bbs")

hsi_cols = spark.table(HSI_V4).columns
spark.sql(f"""SELECT * EXCEPT (_rn) FROM (
  SELECT *, row_number() OVER (PARTITION BY span_id, hazard_id ORDER BY {HSI_DEDUP_ORDER}) AS _rn
  FROM {v3src(HSI_V4, PIN_HSI_V4_SRC)} WHERE span_id NOT LIKE 'BB-%') WHERE _rn = 1""").createOrReplaceTempView("q73_span_rows")
spark.sql(f"SELECT {', '.join(hsi_cols)} FROM q73_span_rows UNION ALL SELECT {', '.join(hsi_cols)} FROM q73_bbs").cache().createOrReplaceTempView("v4_hsi_qa073")

# Gates A-F (BUG-QA-073 acceptance spec, DP P39)
V = "v4_hsi_qa073"
one = lambda sql: spark.sql(sql).collect()[0].asDict()
a = one(f"""SELECT count(*) AS n, sum(int(s.span_id IS NULL)) AS fk_orphans, sum(int(h.span_id LIKE 'BB-%')) AS bb_rows,
  sum(int(h.span_id LIKE 'BBS-%' AND s.conduit_id NOT LIKE 'CDT-BBR-%')) AS bbs_not_bbr FROM {V} h LEFT JOIN {SPAN_V4} s USING (span_id)""")
gate("QA073-A FK clean, 0 BB-*", a["fk_orphans"] == 0 and a["bb_rows"] == 0 and a["bbs_not_bbr"] == 0, a)
b = one(f"SELECT count(*) AS n, count(DISTINCT span_id, hazard_id) AS pairs, count(DISTINCT intersection_id) AS ids, sum(int(hazard_id IS NULL)) AS null_hz FROM {V}")
gate("QA073-B unique pairs and ids", b["n"] == b["pairs"] == b["ids"] and b["null_hz"] == 0, b)
c = one(f"""SELECT sum(int(is_within_radius)) FILTER (WHERE span_id LIKE 'BBS-%') AS bbs_in,
  count(DISTINCT s.parent_route_id, hazard_id) FILTER (WHERE span_id LIKE 'BBS-%' AND is_within_radius) AS route_in,
  sum(int(is_within_radius)) AS total_in FROM {V} LEFT JOIN {SPAN_V4} s USING (span_id)""")
gate("QA073-C BBS in-radius bands", 41800 <= c["bbs_in"] <= 86100 and 15912 <= c["route_in"] <= 36128, c)
kc = ", ".join(x for x in hsi_cols if x not in ("computed_at", "batch_id"))
d = one(f"""SELECT (SELECT count(*) FROM {V} WHERE span_id LIKE 'SPAN-%') AS span_rows,
  (SELECT sum(int(is_within_radius)) FROM {V} WHERE span_id LIKE 'SPAN-%') AS span_in,
  (SELECT count(*) FROM (SELECT {kc} FROM q73_span_rows EXCEPT ALL SELECT {kc} FROM {V} WHERE span_id LIKE 'SPAN-%')) AS d1,
  (SELECT count(*) FROM (SELECT {kc} FROM {V} WHERE span_id LIKE 'SPAN-%' EXCEPT ALL SELECT {kc} FROM q73_span_rows)) AS d2""")
gate("QA073-D SPAN rows unchanged", d == {"span_rows": 181588, "span_in": 93676, "d1": 0, "d2": 0}, d)
e = one(f"""SELECT sum(int(distance_method <> 'geodesic_to_linestring')) AS bad_method,
  sum(int(is_within_radius <> (distance_km <= hazard_radius_km))) AS bad_flag,
  sum(int(proximity_band NOT IN ('direct_impact', 'near_field', 'buffer_zone'))) AS bad_band,
  sum(int(lower(concat_ws('|', hazard_source, hazard_type, affected_service_ids, batch_id)) LIKE '%lumen%')) AS lumen FROM {V}""")
gate("QA073-E semantics", all(v == 0 for v in e.values()), e)
f = {r[0]: r[1] for r in spark.sql(f"""SELECT change_vs_v3, count(DISTINCT s.parent_route_id, hazard_id) FROM {V} JOIN {SPAN_V4} s USING (span_id)
  WHERE span_id LIKE 'BBS-%' GROUP BY 1""").collect()}
f_null = one(f"SELECT sum(int(change_vs_v3 IS NULL)) AS n FROM {V}")["n"]
gate("QA073-F route-grain labels vs v3", f.get("UNCHANGED_IN", 0) + f.get("LOST", 0) == 763 and sum(f.values()) - f.get("GAINED_NEW_PAIR", 0) == 1210 and f_null == 0, f)
g90 = one(f"SELECT count(*) AS span_rows, sum(int(NOT {HSI_ID_VALID})) AS invalid_ids FROM {V} WHERE span_id LIKE 'SPAN-%'")
gate("QA073-G P90 SPAN dedup keeps only formula-valid ids", g90["span_rows"] == 181588 and g90["invalid_ids"] == 0, g90)

if QA073_WRITE and all(g["pass"] for g in GATES if g["gate"].startswith("QA073")):
    assert_v4_target(HSI_V4)
    spark.sql(f"INSERT OVERWRITE {HSI_V4} SELECT * FROM v4_hsi_qa073")
    print("HSI_v4 overwritten from v4_hsi_qa073")
else:
    print("BUG-QA-073 dry run only: no write (needs write_enabled=true, qa073_write_approved=true and all QA073 gates passing)")

In [0]:
# BB-* route rows: rank within route by xxhash64(id), id; segment_seq = pmod(rank-1, segment_count)+1. All other rows identical to v3.
def b5_sql(src: str, id_col: str, pin: int) -> str:
    cols = spark.table(f"{S}.{src}").columns
    sel = ", ".join("g.seg_span_id AS span_id" if c == "span_id" else f"b.{c}" for c in cols)
    return f"""
    WITH seg AS (SELECT parent_route_id, segment_seq, span_id AS seg_span_id FROM v4_span WHERE span_id LIKE 'BBS-%'),
    nseg AS (SELECT parent_route_id, MAX(segment_seq) AS segment_count FROM seg GROUP BY parent_route_id),
    src AS (SELECT * FROM {v3src(src, pin)}),
    bb AS (SELECT s.*, ROW_NUMBER() OVER (PARTITION BY s.span_id ORDER BY xxhash64(s.{id_col}), s.{id_col}) AS rk
           FROM src s WHERE s.span_id LIKE 'BB-%')
    SELECT * FROM src WHERE span_id IS NULL OR span_id NOT LIKE 'BB-%'
    UNION ALL
    SELECT {sel} FROM bb b
    JOIN nseg c ON c.parent_route_id = b.span_id
    JOIN seg g ON g.parent_route_id = b.span_id AND g.segment_seq = pmod(b.rk - 1, c.segment_count) + 1"""


spark.sql(b5_sql("resource_alarm", "alarm_id", PIN_ALARM)).createOrReplaceTempView("v4_alarm")
spark.sql(b5_sql("resource_performance_measure", "measure_id", PIN_PERF)).createOrReplaceTempView("v4_perf")

for v, pin, src in (("v4_alarm", PIN_ALARM, "resource_alarm"), ("v4_perf", PIN_PERF, "resource_performance_measure")):
    g = spark.sql(f"""SELECT (SELECT count(*) FROM {v}) n_v4, (SELECT count(*) FROM {v3src(src, pin)}) n_v3,
      (SELECT count(*) FROM {v} WHERE span_id LIKE 'BB-%' AND span_id NOT LIKE 'BBS-%') n_bb_left,
      (SELECT count(*) FROM {v} a LEFT ANTI JOIN v4_span s USING (span_id) WHERE a.span_id IS NOT NULL) fk_orphans,
      (SELECT max(c) / min(c) FROM (SELECT span_id, count(*) c FROM {v} WHERE span_id LIKE 'BBS-%' GROUP BY 1)) bal""").collect()[0]
    gate(f"B5 {src}: same rows, 0 BB-* left, 0 FK orphans", g.n_v4 == g.n_v3 and g.n_bb_left == 0 and g.fk_orphans == 0, g.asDict())

if WRITE_ENABLED:
    assert_v4_target(ALARM_V4)
    assert_v4_target(PERF_V4)
    spark.sql(f"CREATE TABLE IF NOT EXISTS {ALARM_V4} AS SELECT * FROM v4_alarm WHERE 1 = 0")
    spark.sql(f"INSERT OVERWRITE {ALARM_V4} SELECT * FROM v4_alarm")
    spark.sql(f"CREATE TABLE IF NOT EXISTS {PERF_V4} AS SELECT * FROM v4_perf WHERE 1 = 0")
    spark.sql(f"INSERT OVERWRITE {PERF_V4} SELECT * FROM v4_perf")
diff_vs_live("v4_alarm", ALARM_V4, "alarm_id", "B5 alarm_v4")
diff_vs_live("v4_perf", PERF_V4, "measure_id", "B5 perf_v4")

In [0]:
# Verbatim DDL (SHOW CREATE TABLE form). Dry run checks the live definition is byte-identical; write mode re-creates the view.
MV_MRC_V4_DDL = r"""CREATE VIEW cdm_tmforum.provable_resilience_intelligence.mv_mrc_protected_v4_candidate (
  Service Type COMMENT 'Service technology type (wavelength, ethernet, dark_fiber, ip_vpn, managed_security)',
  SLA Tier COMMENT 'Service SLA tier (standard, premium, premium_plus)',
  Sold as Diverse COMMENT 'Whether the service was sold with a path diversity guarantee',
  Customer Tier COMMENT 'Customer tier: platinum, gold, silver',
  Industry COMMENT 'Customer industry vertical',
  Region COMMENT 'Customer region',
  Has Completed Action COMMENT 'True if the service has at least one completed proactive_action of any type',
  MRC Protected USD COMMENT 'Business KPI. Monthly recurring charge summed over DISTINCT services with at least one completed proactive_action (any action type).',
  Services Protected COMMENT 'Distinct services with at least one completed proactive_action',
  MRC Protected Proactive Only USD COMMENT 'Supporting measure. MRC over distinct services with at least one completed reroute or preposition_splice action.',
  Services Protected Proactive Only COMMENT 'Distinct services with at least one completed reroute or preposition_splice action',
  Total MRC USD COMMENT 'Total monthly recurring charge across all services in scope (denominator)',
  MRC Protected Share COMMENT 'Share of total MRC on services with at least one completed proactive_action',
  Total Services COMMENT 'Distinct services in scope')
COMMENT 'V4 CANDIDATE (DEC-PRI-REV-PROTECTED-DEFINITION, P29-DA-REV-PROTECTED-DEDUP). Grain: one row per Lakelink Fiber service (service_id). MRC Protected = SUM of monthly recurring charge (mrc_usd) over DISTINCT services with at least one completed proactive_action. Deduplicated per service, so it never double-counts services with multiple actions. Replaces the retired \'Revenue Protected USD\' (action_queue_status) and \'revenue_protected\' (prevented_outages) at the v4 cutover. Frozen v3 metric views are unchanged.'
WITH METRICS
LANGUAGE YAML
AS
$$
version: 1.1

source: |-
  SELECT si.service_id, si.service_type, si.sla_tier, si.sold_as_diverse, si.mrc_usd,
         c.customer_id, c.tier AS customer_tier, c.industry, c.region,
         COALESCE(pa.completed_actions, 0) AS completed_actions,
         COALESCE(pa.total_actions, 0) AS total_actions,
         COALESCE(pa.has_completed_action, 0) AS has_completed_action,
         COALESCE(pa.has_completed_proactive, 0) AS has_completed_proactive
  FROM cdm_tmforum.provable_resilience_intelligence.service_inventory si
  LEFT JOIN cdm_tmforum.provable_resilience_intelligence.customer c ON si.customer_id = c.customer_id
  LEFT JOIN (
    SELECT service_id,
           COUNT(*) AS total_actions,
           COUNT_IF(status = 'completed') AS completed_actions,
           MAX(CASE WHEN status = 'completed' THEN 1 ELSE 0 END) AS has_completed_action,
           MAX(CASE WHEN status = 'completed' AND action_type IN ('reroute','preposition_splice') THEN 1 ELSE 0 END) AS has_completed_proactive
    FROM cdm_tmforum.provable_resilience_intelligence.proactive_action
    GROUP BY service_id) pa ON si.service_id = pa.service_id

comment: "V4 CANDIDATE (DEC-PRI-REV-PROTECTED-DEFINITION, P29-DA-REV-PROTECTED-DEDUP).\
  \ Grain: one row per Lakelink Fiber service (service_id). MRC Protected = SUM of\
  \ monthly recurring charge (mrc_usd) over DISTINCT services with at least one completed\
  \ proactive_action. Deduplicated per service, so it never double-counts services\
  \ with multiple actions. Replaces the retired 'Revenue Protected USD' (action_queue_status)\
  \ and 'revenue_protected' (prevented_outages) at the v4 cutover. Frozen v3 metric\
  \ views are unchanged."

dimensions:
  - name: Service Type
    expr: service_type
    comment: "Service technology type (wavelength, ethernet, dark_fiber, ip_vpn, managed_security)"

  - name: SLA Tier
    expr: sla_tier
    comment: "Service SLA tier (standard, premium, premium_plus)"

  - name: Sold as Diverse
    expr: sold_as_diverse
    comment: Whether the service was sold with a path diversity guarantee

  - name: Customer Tier
    expr: customer_tier
    comment: "Customer tier: platinum, gold, silver"

  - name: Industry
    expr: industry
    comment: Customer industry vertical

  - name: Region
    expr: region
    comment: Customer region

  - name: Has Completed Action
    expr: has_completed_action = 1
    comment: True if the service has at least one completed proactive_action of any
      type

measures:
  - name: MRC Protected USD
    expr: SUM(mrc_usd) FILTER (WHERE has_completed_action = 1)
    comment: Business KPI. Monthly recurring charge summed over DISTINCT services
      with at least one completed proactive_action (any action type).
    display_name: MRC Protected
    format:
      type: currency
      currency_code: USD
      decimal_places:
        type: exact
        places: 0
      abbreviation: compact
    synonyms:
      - revenue protected
      - mrc protected
      - protected revenue
      - revenue saved

  - name: Services Protected
    expr: COUNT(DISTINCT service_id) FILTER (WHERE has_completed_action = 1)
    comment: Distinct services with at least one completed proactive_action

  - name: MRC Protected Proactive Only USD
    expr: SUM(mrc_usd) FILTER (WHERE has_completed_proactive = 1)
    comment: Supporting measure. MRC over distinct services with at least one completed
      reroute or preposition_splice action.
    format:
      type: currency
      currency_code: USD
      decimal_places:
        type: exact
        places: 0
      abbreviation: compact

  - name: Services Protected Proactive Only
    expr: COUNT(DISTINCT service_id) FILTER (WHERE has_completed_proactive = 1)
    comment: Distinct services with at least one completed reroute or preposition_splice
      action

  - name: Total MRC USD
    expr: SUM(mrc_usd)
    comment: Total monthly recurring charge across all services in scope (denominator)

  - name: MRC Protected Share
    expr: "SUM(mrc_usd) FILTER (WHERE has_completed_action = 1) / NULLIF(SUM(mrc_usd),\
      \ 0)"
    comment: Share of total MRC on services with at least one completed proactive_action
    format:
      type: percentage
      decimal_places:
        type: exact
        places: 1

  - name: Total Services
    expr: COUNT(DISTINCT service_id)
    comment: Distinct services in scope
$$
"""

MV_HPC_V4_DDL = r"""CREATE VIEW cdm_tmforum.provable_resilience_intelligence.hazard_proximity_count_v4_candidate (
  hazard_type COMMENT 'Type of geographic hazard (wildfire, flood, severe_weather, dig_ticket, earthquake)',
  severity COMMENT 'Hazard severity (critical, high, medium, low)',
  source COMMENT 'Data source for hazard event (noaa, usgs, 811_system, nifc)',
  is_active COMMENT 'Whether hazard is currently active (no end date) or resolved',
  event_month COMMENT 'Month when hazard event started',
  hazard_proximity_count COMMENT 'Total distinct geographic hazard events',
  active_hazard_count COMMENT 'Currently active (unresolved) hazard events',
  critical_hazard_count COMMENT 'Hazard events with critical severity',
  avg_radius_km COMMENT 'Average hazard impact radius in kilometers',
  max_radius_km COMMENT 'Maximum hazard impact radius — indicates worst-case exposure area',
  dig_ticket_count COMMENT '811 dig ticket events — excavation threats to buried fiber')
COMMENT 'V4 CANDIDATE of hazard_proximity_count for Lakelink Fiber. Excludes 1,036 future-dated (+5y shifted) wildfire events via filter event_start <= current_timestamp(); active = no end date or end in the future. Promote at v4 cutover.'
WITH METRICS
LANGUAGE YAML
AS
$$
version: 1.1

source: cdm_tmforum.provable_resilience_intelligence.geographic_hazard_event

filter: event_start <= current_timestamp()

comment: "V4 CANDIDATE of hazard_proximity_count for Lakelink Fiber. Excludes 1,036\
  \ future-dated (+5y shifted) wildfire events via filter event_start <= current_timestamp();\
  \ active = no end date or end in the future. Promote at v4 cutover."

dimensions:
  - name: hazard_type
    expr: hazard_type
    comment: "Type of geographic hazard (wildfire, flood, severe_weather, dig_ticket,\
      \ earthquake)"
    display_name: Hazard Type
    synonyms:
      - threat type
      - hazard category
      - event type

  - name: severity
    expr: severity
    comment: "Hazard severity (critical, high, medium, low)"
    display_name: Severity
    synonyms:
      - hazard severity
      - threat level

  - name: source
    expr: source
    comment: "Data source for hazard event (noaa, usgs, 811_system, nifc)"
    display_name: Source System

  - name: is_active
    expr: CASE WHEN event_end IS NULL OR event_end > current_timestamp() THEN 'active'
      ELSE 'resolved' END
    comment: Whether hazard is currently active (no end date) or resolved
    display_name: Hazard Status
    synonyms:
      - active status
      - current threats

  - name: event_month
    expr: "DATE_TRUNC('MONTH', event_start)"
    comment: Month when hazard event started
    display_name: Event Month
    format:
      type: date
      date_format: locale_short_month

measures:
  - name: hazard_proximity_count
    expr: COUNT(DISTINCT hazard_id)
    comment: Total distinct geographic hazard events
    display_name: Hazard Count
    synonyms:
      - number of hazards
      - threat count
      - hazard proximity count

  - name: active_hazard_count
    expr: COUNT(DISTINCT CASE WHEN event_end IS NULL OR event_end > current_timestamp()
      THEN hazard_id END)
    comment: Currently active (unresolved) hazard events
    display_name: Active Hazards
    synonyms:
      - current threats
      - ongoing hazards

  - name: critical_hazard_count
    expr: COUNT(DISTINCT CASE WHEN severity = 'critical' THEN hazard_id END)
    comment: Hazard events with critical severity
    display_name: Critical Hazards

  - name: avg_radius_km
    expr: AVG(radius_km)
    comment: Average hazard impact radius in kilometers
    display_name: Avg Radius (km)
    format:
      type: number
      decimal_places:
        type: exact
        places: 1

  - name: max_radius_km
    expr: MAX(radius_km)
    comment: Maximum hazard impact radius — indicates worst-case exposure area
    display_name: Max Radius (km)
    format:
      type: number
      decimal_places:
        type: exact
        places: 1

  - name: dig_ticket_count
    expr: COUNT(DISTINCT CASE WHEN hazard_type = 'dig_ticket' THEN hazard_id END)
    comment: 811 dig ticket events — excavation threats to buried fiber
    display_name: Dig Ticket Count
    synonyms:
      - 811 tickets
      - excavation threats
$$
"""

for name, ddl in ((MV_MRC_V4, MV_MRC_V4_DDL), (MV_HPC_V4, MV_HPC_V4_DDL)):
    assert_v4_target(name)
    assert ddl.startswith("CREATE VIEW " + name + " ("), "DDL target mismatch for " + name
    if WRITE_ENABLED:
        spark.sql(ddl.replace("CREATE VIEW", "CREATE OR REPLACE VIEW", 1))
    live = spark.sql("SHOW CREATE TABLE " + name).collect()[0][0]  # read only
    gate(f"metric view {name.split('.')[-1]} DDL == live", live.strip() == ddl.strip(), f"{len(ddl.strip())} chars vs live {len(live.strip())}")

In [0]:
# P52-DE-V4-PROACTIVE-ACTION-CANDIDATE (runbook step g)
# proactive_action_v4_candidate = proactive_action@15 (all 11,276 v3 rows, BB-* span_id remapped to BBS-*)
#                                + ACT-V4-* rows from ACT-RULE-v1 applied to resilience_score_snapshot_v4_candidate.
# ACT-RULE-v1 is the rule that produced the 475 ACT-V22-* rows in proactive_action@15. It reproduces them 475/475 with 0 diffs:
#   critical (overall_score < 40) AND no existing 'pending' action for the service ->
#   weakest driver = telemetry (strictly lowest) -> dispatch_crew / Field-Ops-Dispatch (DSP), otherwise preposition_splice / Plant-Engineering (PRE).
#   Revenue in trigger_reason uses HALF_EVEN rounding (bround), as in the V22 write.
# Write needs write_enabled=true AND p52_write_approved=true. The second flag stays false until the P36-ML v4 rescore has landed.
PA_V4 = f"{S}.proactive_action_v4_candidate"
assert_v4_target(PA_V4)
# P67 / BUG-QA-095: PIN_* are fingerprint keys / lineage labels. v3src() time-travels (fallback only) to
# demo_freeze_manifest.restore_version: proactive_action 15, resilience_score_snapshot 24, service_path 11.
PIN_PA = 15                                # proactive_action@15 (last data commit, V22 rows); restore_version 15
PIN_PA_PRE_V22 = 14                        # proactive_action@14 = restore_version 15 + PIN_FILTER (no ACT-V22-*)
PIN_SNAP_V3 = 22                           # resilience_score_snapshot@22 (lineage); restore_version 24
PIN_SP = 11                                # service_path@11 (FREEZE-v3 label, same as cell 7c; BUG-QA-089); restore_version 11
print("P52 restore_version:", {t: restore_version(t) for t in ("proactive_action", "resilience_score_snapshot", "service_path")})
V22_BUILD_TS = "2026-10-24 17:13:14.023839"
SNAP_V4 = f"{S}.resilience_score_snapshot_v4_candidate"
dbutils.widgets.dropdown("p52_write_approved", "false", ["false", "true"], "P52: P36 v4 rescore landed")
P52_WRITE_APPROVED = dbutils.widgets.get("p52_write_approved") == "true"


def act_rule_v1_sql(snap_view, prior_view, label, tag, created_at_expr, hop1_view):
    return f"""
WITH base AS (
  SELECT s.service_id, s.overall_score, s.path_diversity_score, s.hazard_exposure_score, s.telemetry_health_score,
         s.active_hazards, s.srlg_count, s.revenue_at_risk_usd, si.sla_tier,
         CASE WHEN s.telemetry_health_score < s.hazard_exposure_score AND s.telemetry_health_score < s.path_diversity_score THEN 'telemetry'
              WHEN s.path_diversity_score < s.hazard_exposure_score AND s.path_diversity_score < s.telemetry_health_score THEN 'path'
              ELSE 'hazard' END AS weakest
  FROM {snap_view} s JOIN {S}.service_inventory si ON si.service_id = s.service_id
  WHERE s.overall_score < 40
    AND NOT EXISTS (SELECT 1 FROM {prior_view} p WHERE p.service_id = s.service_id AND p.status = 'pending'))
SELECT concat('ACT-{tag}-', CASE WHEN weakest = 'telemetry' THEN 'DSP' ELSE 'PRE' END, '-', b.service_id) AS action_id,
  b.service_id,
  CASE WHEN weakest = 'telemetry' THEN 'dispatch_crew' ELSE 'preposition_splice' END AS action_type,
  concat('Critical score ', format_number(b.overall_score, 1), ' ({label}). Weakest driver: ', weakest,
    ' (path ', format_number(b.path_diversity_score, 1), ', hazard ', format_number(b.hazard_exposure_score, 1),
    ', telemetry ', format_number(b.telemetry_health_score, 1), '). ', b.active_hazards, ' active hazards, ',
    b.srlg_count, ' shared SRLGs. ', initcap(b.sla_tier), ' SLA. Revenue at risk: $',
    format_number(bround(b.revenue_at_risk_usd, 0), 0)) AS trigger_reason,
  'critical' AS priority, 'pending' AS status,
  CASE WHEN weakest = 'telemetry' THEN 'Field-Ops-Dispatch' ELSE 'Plant-Engineering' END AS assigned_to,
  {created_at_expr} AS created_at, CAST(NULL AS TIMESTAMP) AS resolved_at, CAST(NULL AS STRING) AS resolution_notes,
  h.span_id, 'DERIVED' AS origin
FROM base b LEFT JOIN {hop1_view} h ON h.service_id = b.service_id"""


spark.sql(f"CREATE OR REPLACE TEMP VIEW p52_pa14 AS SELECT * FROM {v3src('proactive_action', PIN_PA_PRE_V22)}")
spark.sql(f"CREATE OR REPLACE TEMP VIEW p52_pa15 AS SELECT * FROM {v3src('proactive_action', PIN_PA)}")
spark.sql(f"CREATE OR REPLACE TEMP VIEW p52_snap3 AS SELECT * FROM {v3src('resilience_score_snapshot', PIN_SNAP_V3)}")
spark.sql(f"CREATE OR REPLACE TEMP VIEW p52_hop1_v3 AS SELECT service_id, span_id FROM {v3src('service_path', PIN_SP)} WHERE path_type = 'working' AND hop_sequence = 1")

# G1: the rule reproduces the v3 V22 rows exactly
spark.sql(act_rule_v1_sql("p52_snap3", "p52_pa14", "snapshot v22, BUG-QA-060/061 rescore", "V22",
                          f"TIMESTAMP'{V22_BUILD_TS}'", "p52_hop1_v3")).createOrReplaceTempView("p52_repro_v22")
g1 = spark.sql("""SELECT
  (SELECT count(*) FROM (SELECT * FROM p52_repro_v22 EXCEPT ALL SELECT * FROM p52_pa15 WHERE action_id LIKE 'ACT-V22-%')) AS extra,
  (SELECT count(*) FROM (SELECT * FROM p52_pa15 WHERE action_id LIKE 'ACT-V22-%' EXCEPT ALL SELECT * FROM p52_repro_v22)) AS missing,
  (SELECT count(*) FROM p52_repro_v22) AS n""").collect()[0]
gate("P52-G1 ACT-RULE-v1 reproduces the 475 V22 rows", g1.extra == 0 and g1.missing == 0 and g1.n == 475, g1.asDict())

# BUG-QA-132: working hop 1 BB-<route> -> BBS-<route>-NNN uses the P54 orientation rule (cell 7c, DEC-PRI-V4-HOP1-BBS-REMAP),
# so proactive_action_v4.span_id == service_path_v4 working hop_sequence 1. Hop 1 never has a previous hop, so:
#   next hop exists -> exit = route endpoint shared with the next hop (NEXT_HOP_JUNCTION)
#   else            -> entry = route endpoint nearer the service A-end site, haversine (NEAREST_ENDPOINT_TO_A_END)
# First segment in traversal order: AZ -> segment_seq 1, ZA -> segment_seq = segment_count.
# Replaces the old midpoint-nearest heuristic, which disagreed with P54 hop 1 for 1,307 of 1,973 BB hop-1 PA services
# (DE S97 dry run). The independent gate BUG-QA-132 is in cell 7c, against the fully expanded v4_service_path.
_p52_hav = lambda la1, lo1, la2, lo2: (f"2*{R_KM}*asin(sqrt(pow(sin(radians({la2}-{la1})/2),2)+"
                                       f"cos(radians({la1}))*cos(radians({la2}))*pow(sin(radians({lo2}-{lo1})/2),2)))")
spark.sql(f"CREATE OR REPLACE TEMP VIEW p52_sp3w AS SELECT * FROM {v3src('service_path', PIN_SP)} WHERE path_type = 'working'")
spark.sql(f"CREATE OR REPLACE TEMP VIEW p52_span3 AS SELECT span_id, site_a_id, site_z_id FROM {v3src('physical_resource_span', PIN_SPAN)}")
spark.sql(f"""CREATE OR REPLACE TEMP VIEW p52_hop1_orient AS
WITH h AS (
  SELECT p.path_id, p.service_id, p.span_id, p.hop_sequence, s.site_a_id AS ra, s.site_z_id AS rz,
         lead(p.span_id) OVER (PARTITION BY p.path_id ORDER BY p.hop_sequence) AS next_span
  FROM p52_sp3w p JOIN p52_span3 s ON s.span_id = p.span_id)
SELECT h.path_id, h.service_id, h.span_id AS v3_span_id,
  CASE WHEN h.span_id NOT LIKE 'BB-%' THEN CAST(NULL AS STRING)
       WHEN h.next_span IS NOT NULL THEN
         CASE WHEN h.rz IN (nx.site_a_id, nx.site_z_id) AND h.ra NOT IN (nx.site_a_id, nx.site_z_id) THEN 'AZ'
              WHEN h.ra IN (nx.site_a_id, nx.site_z_id) AND h.rz NOT IN (nx.site_a_id, nx.site_z_id) THEN 'ZA' END
       ELSE CASE WHEN {_p52_hav('sa.latitude', 'sa.longitude', 'ga.latitude', 'ga.longitude')}
                   <= {_p52_hav('sa.latitude', 'sa.longitude', 'gz.latitude', 'gz.longitude')} THEN 'AZ' ELSE 'ZA' END END AS traverse_dir
FROM h LEFT JOIN p52_span3 nx ON nx.span_id = h.next_span
  JOIN {S}.service_inventory si ON si.service_id = h.service_id
  JOIN {S}.geographic_site sa ON sa.site_id = si.site_a_id
  JOIN {S}.geographic_site ga ON ga.site_id = h.ra
  JOIN {S}.geographic_site gz ON gz.site_id = h.rz
WHERE h.hop_sequence = 1""")
spark.sql(f"""CREATE OR REPLACE TEMP VIEW p52_hop1_v4 AS
SELECT o.service_id, COALESCE(g.span_id, o.v3_span_id) AS span_id
FROM p52_hop1_orient o LEFT JOIN {SPAN_V4} g
  ON g.parent_route_id = o.v3_span_id AND g.segment_seq = CASE WHEN o.traverse_dir = 'AZ' THEN 1 ELSE g.segment_count END""")
g132 = spark.sql("""SELECT count(*) AS n, count(DISTINCT service_id) AS svcs,
    count_if(span_id IS NULL) AS null_span, count_if(span_id LIKE 'BB-%') AS bb_left,
    (SELECT count_if(v3_span_id LIKE 'BB-%' AND traverse_dir IS NULL) FROM p52_hop1_orient) AS unresolved,
    (SELECT count_if(traverse_dir = 'AZ') FROM p52_hop1_orient) AS az, (SELECT count_if(traverse_dir = 'ZA') FROM p52_hop1_orient) AS za
  FROM p52_hop1_v4""").collect()[0]
gate("BUG-QA-132a hop-1 remap: 1 row per service, 0 NULL, 0 BB-*, 0 unresolved, 1,143 AZ + 1,253 ZA (= P54-G7)",
     g132.n == g132.svcs and g132.null_span == 0 and g132.bb_left == 0 and g132.unresolved == 0
     and g132.az == 1143 and g132.za == 1253, g132.asDict())

snap4_ts = spark.sql(f"SELECT max(score_time) AS t FROM {SNAP_V4}").collect()[0].t
spark.sql(act_rule_v1_sql(SNAP_V4, "p52_pa15", "snapshot v4 candidate, PRI v4 rescore", "V4",
                          f"TIMESTAMP'{snap4_ts}'", "p52_hop1_v4")).createOrReplaceTempView("p52_pa4_new")
# BUG-QA-128 fix: v3 proactive_action.span_id is off-path for 10,800/10,801 rows (only 476 match working hop-1).
# Root cause: the original P6-ACTION-SPAN-FK backfill picked a span randomly, not hop-1 working.
# Fix: always take the v4 working hop-1 span via p52_hop1_v4 (which already includes the BB→BBS remap).
# This corrects all carried v3 rows, not just the BB-* remap. Verified: 0 FK orphans, 0 off-path after fix.
spark.sql("""CREATE OR REPLACE TEMP VIEW p52_pa4_carry AS
SELECT p.action_id, p.service_id, p.action_type, p.trigger_reason, p.priority, p.status, p.assigned_to, p.created_at,
       p.resolved_at, p.resolution_notes, h.span_id, p.origin
FROM p52_pa15 p LEFT JOIN p52_hop1_v4 h ON h.service_id = p.service_id""")
spark.sql("CREATE OR REPLACE TEMP VIEW v4_proactive_action AS SELECT * FROM p52_pa4_carry UNION ALL SELECT * FROM p52_pa4_new")

q = {
    "n": "SELECT count(*) FROM v4_proactive_action",
    "n_new": "SELECT count(*) FROM p52_pa4_new",
    "uniq": "SELECT count(DISTINCT action_id) FROM v4_proactive_action",
    "bb_left": "SELECT count(*) FROM v4_proactive_action WHERE span_id LIKE 'BB-%'",
    "span_orphans": f"SELECT count(*) FROM v4_proactive_action p LEFT ANTI JOIN {SPAN_V4} s ON s.span_id = p.span_id WHERE p.span_id IS NOT NULL",
    "svc_orphans": f"SELECT count(*) FROM v4_proactive_action p LEFT ANTI JOIN {S}.service_inventory s ON s.service_id = p.service_id",
    "null_span": "SELECT count(*) FROM v4_proactive_action WHERE span_id IS NULL",
    "crit_uncovered": f"""SELECT count(*) FROM {SNAP_V4} s LEFT ANTI JOIN (SELECT DISTINCT service_id FROM v4_proactive_action
                          WHERE status = 'pending') p ON p.service_id = s.service_id WHERE s.overall_score < 40""",
    "carry_non_span_changed": """SELECT count(*) FROM (
        SELECT * EXCEPT (span_id) FROM p52_pa15 EXCEPT ALL SELECT * EXCEPT (span_id) FROM p52_pa4_carry)""",
}
g = {k: spark.sql(v).collect()[0][0] for k, v in q.items()}
gate("P52-G2 PK unique, 0 BB-* left, 0 span/service FK orphans, 0 NULL span",
     g["uniq"] == g["n"] and g["bb_left"] == 0 and g["span_orphans"] == 0 and g["svc_orphans"] == 0 and g["null_span"] == 0, g)
gate("P52-G3 every critical v4 service has a pending action", g["crit_uncovered"] == 0, g["crit_uncovered"])
gate("P52-G4 v3 rows carried unchanged except span_id (BUG-QA-128 hop-1 fix + BB remap)", g["carry_non_span_changed"] == 0 and g["n"] - g["n_new"] == 11276, g)

# BUG-QA-128 gate: every action's span_id must be its service's working hop-1 span (v4).
# Validates both carried v3 rows (hop-1 fix) and new ACT-V4-* rows (rule already correct).
# Note (BUG-QA-132): this joins to p52_hop1_v4, the same source as span_id, so it is an internal consistency check only.
# The independent check (vs the fully expanded v4_service_path working hop_sequence 1) is gate BUG-QA-132 in cell 7c.
g128 = spark.sql("""SELECT count(*) AS off_hop1 FROM v4_proactive_action pa
    LEFT ANTI JOIN p52_hop1_v4 h ON h.service_id = pa.service_id AND h.span_id = pa.span_id""").collect()[0].off_hop1
gate("BUG-QA-128 all v4 span_ids are working hop-1", g128 == 0, {"off_hop1": g128})

if WRITE_ENABLED and P52_WRITE_APPROVED:
    assert_v4_target(PA_V4)
    spark.sql(f"CREATE TABLE IF NOT EXISTS {PA_V4} AS SELECT * FROM v4_proactive_action WHERE 1 = 0")
    spark.sql(f"INSERT OVERWRITE {PA_V4} SELECT * FROM v4_proactive_action")
elif spark.catalog.tableExists(PA_V4):
    diff_vs_live("v4_proactive_action", PA_V4, "action_id", "P52 proactive_action_v4")
else:
    print(f"P52 dry run: {PA_V4} not built yet (waits on P36-ML rescore). Preview rows: {g['n']} ({g['n_new']} new ACT-V4-*).")

In [0]:
# P54-DE-SERVICE-PATH-V4-WRITER (runbook B3), spec P53-DP-SERVICE-PATH-V4-HOP1-SPEC, decision DEC-PRI-V4-HOP1-BBS-REMAP.
# service_path_v4_candidate = service_path@11 with every BB-<route> hop (any position) expanded into that route's
# BBS-<route>-NNN segments from physical_resource_span_v4_candidate, oriented by topology:
#   prev hop exists -> entry = route endpoint shared with prev hop (PREV_HOP_JUNCTION)
#   else next hop   -> exit  = route endpoint shared with next hop (NEXT_HOP_JUNCTION)
#   else            -> entry = route endpoint nearer the service A-end site, haversine (NEAREST_ENDPOINT_TO_A_END)
# AZ: sub_ord = segment_seq; ZA: sub_ord = segment_count + 1 - segment_seq. hop_sequence = row_number over (hop_v3, sub_ord).
# SPAN-* rows are carried unchanged (span_remap_method UNCHANGED_V3, traverse_dir NULL). created_at is the v3 value on every row.
# Consumers that count hops must count DISTINCT v3_span_id (or parent_route_id), not rows: the table grows ~30x.
# Write needs write_enabled=true AND p54_write_approved=true (v4 cutover window, DEC-PRI-V4-CUTOVER-TARGET-1030).
SP_V4 = f"{S}.service_path_v4_candidate"
assert_v4_target(SP_V4)
# P67 / BUG-QA-095: PIN_* are fingerprint keys / lineage labels; v3src() falls back to manifest restore_version.
PIN_SP = 11                                # service_path@11 (FREEZE-v3 label; 54,572 rows); restore_version 11
print("P54 restore_version:", {t: restore_version(t) for t in ("service_path", "physical_resource_span")})
P54_EXPECT_ROWS = 1655121                  # P53 spec gate
dbutils.widgets.dropdown("p54_write_approved", "false", ["false", "true"], "P54: v4 cutover window open")
P54_WRITE_APPROVED = dbutils.widgets.get("p54_write_approved") == "true"

HAV_SQL = lambda la1, lo1, la2, lo2: (f"2*{R_KM}*asin(sqrt(pow(sin(radians({la2}-{la1})/2),2)+"
                                      f"cos(radians({la1}))*cos(radians({la2}))*pow(sin(radians({lo2}-{lo1})/2),2)))")
spark.sql(f"CREATE OR REPLACE TEMP VIEW p54_sp3 AS SELECT * FROM {v3src('service_path', PIN_SP)}")
spark.sql(f"CREATE OR REPLACE TEMP VIEW p54_span3 AS SELECT span_id, site_a_id, site_z_id FROM {v3src('physical_resource_span', PIN_SPAN)}")
spark.sql(f"""CREATE OR REPLACE TEMP VIEW p54_orient AS
WITH h AS (
  SELECT p.*, s.site_a_id AS ra, s.site_z_id AS rz,
         lag(p.span_id) OVER (PARTITION BY p.path_id ORDER BY p.hop_sequence) AS prev_span,
         lead(p.span_id) OVER (PARTITION BY p.path_id ORDER BY p.hop_sequence) AS next_span
  FROM p54_sp3 p JOIN p54_span3 s ON s.span_id = p.span_id),
bb AS (
  SELECT h.*, pv.site_a_id AS pa, pv.site_z_id AS pz, nx.site_a_id AS na, nx.site_z_id AS nz,
         sa.latitude AS alat, sa.longitude AS alon, ga.latitude AS ra_lat, ga.longitude AS ra_lon,
         gz.latitude AS rz_lat, gz.longitude AS rz_lon
  FROM h LEFT JOIN p54_span3 pv ON pv.span_id = h.prev_span
         LEFT JOIN p54_span3 nx ON nx.span_id = h.next_span
         JOIN {S}.service_inventory si ON si.service_id = h.service_id
         JOIN {S}.geographic_site sa ON sa.site_id = si.site_a_id
         JOIN {S}.geographic_site ga ON ga.site_id = h.ra
         JOIN {S}.geographic_site gz ON gz.site_id = h.rz
  WHERE h.span_id LIKE 'BB-%')
SELECT path_id, service_id, path_type, span_id AS bb_span_id, hop_sequence AS hop_v3, created_at,
  CASE WHEN prev_span IS NOT NULL THEN
         CASE WHEN ra IN (pa, pz) AND rz NOT IN (pa, pz) THEN 'AZ' WHEN rz IN (pa, pz) AND ra NOT IN (pa, pz) THEN 'ZA' END
       WHEN next_span IS NOT NULL THEN
         CASE WHEN rz IN (na, nz) AND ra NOT IN (na, nz) THEN 'AZ' WHEN ra IN (na, nz) AND rz NOT IN (na, nz) THEN 'ZA' END
       ELSE CASE WHEN {HAV_SQL('alat', 'alon', 'ra_lat', 'ra_lon')} <= {HAV_SQL('alat', 'alon', 'rz_lat', 'rz_lon')}
                 THEN 'AZ' ELSE 'ZA' END END AS traverse_dir,
  CASE WHEN prev_span IS NOT NULL THEN 'PREV_HOP_JUNCTION' WHEN next_span IS NOT NULL THEN 'NEXT_HOP_JUNCTION'
       ELSE 'NEAREST_ENDPOINT_TO_A_END' END AS span_remap_method
FROM bb""")
spark.sql(f"""CREATE OR REPLACE TEMP VIEW p54_rows AS
SELECT p.path_id, p.service_id, p.path_type, p.span_id, p.hop_sequence AS hop_v3, p.span_id AS v3_span_id,
       1 AS sub_ord, CAST(NULL AS STRING) AS traverse_dir, 'UNCHANGED_V3' AS span_remap_method, p.created_at, p.origin
FROM p54_sp3 p WHERE p.span_id NOT LIKE 'BB-%'
UNION ALL
SELECT o.path_id, o.service_id, o.path_type, g.span_id, o.hop_v3, o.bb_span_id,
       CASE WHEN o.traverse_dir = 'AZ' THEN g.segment_seq ELSE g.segment_count + 1 - g.segment_seq END,
       o.traverse_dir, o.span_remap_method, o.created_at, 'DERIVED'
FROM p54_orient o JOIN {SPAN_V4} g ON g.parent_route_id = o.bb_span_id""")
spark.sql("""CREATE OR REPLACE TEMP VIEW v4_service_path AS
SELECT concat(path_id, '-', lpad(CAST(hop_sequence AS STRING), 4, '0')) AS path_hop_id,
       path_id, service_id, path_type, span_id, hop_sequence, hop_v3, v3_span_id, sub_ord, traverse_dir,
       span_remap_method, created_at, origin
FROM (SELECT *, CAST(row_number() OVER (PARTITION BY path_id ORDER BY hop_v3, sub_ord) AS INT) AS hop_sequence FROM p54_rows)""")

q = {
    "n": "SELECT count(*) FROM v4_service_path",
    "paths": "SELECT count(DISTINCT path_id) FROM v4_service_path",
    "svcs": "SELECT count(DISTINCT service_id) FROM v4_service_path",
    "paths_v3": "SELECT count(DISTINCT path_id) FROM p54_sp3",
    "svcs_v3": "SELECT count(DISTINCT service_id) FROM p54_sp3",
    "expect_n": f"""SELECT (SELECT count(*) FROM p54_sp3 WHERE span_id NOT LIKE 'BB-%') + (SELECT sum(g.segment_count) FROM p54_sp3 p
                   JOIN (SELECT DISTINCT parent_route_id, segment_count FROM {SPAN_V4} WHERE parent_route_id IS NOT NULL) g
                   ON g.parent_route_id = p.span_id)""",
    "bb_left": "SELECT count(*) FROM v4_service_path WHERE span_id LIKE 'BB-%'",
    "span_orphans": f"SELECT count(*) FROM v4_service_path p LEFT ANTI JOIN {SPAN_V4} s ON s.span_id = p.span_id",
    "svc_orphans": f"SELECT count(*) FROM v4_service_path p LEFT ANTI JOIN {S}.service_inventory s ON s.service_id = p.service_id",
    "dup_pk": "SELECT count(*) - count(DISTINCT path_id, hop_sequence) FROM v4_service_path",
    "dup_path_hop_id": "SELECT count(*) - count(DISTINCT path_hop_id) FROM v4_service_path",
    "max_hop": "SELECT max(hop_sequence) FROM v4_service_path",
    "hop_gaps": "SELECT count(*) FROM (SELECT path_id FROM v4_service_path GROUP BY path_id HAVING min(hop_sequence) <> 1 OR max(hop_sequence) <> count(*))",
    "span_extra": """SELECT count(*) FROM (SELECT path_id, service_id, path_type, span_id, hop_v3, created_at, origin FROM v4_service_path
                     WHERE span_remap_method = 'UNCHANGED_V3' EXCEPT ALL
                     SELECT path_id, service_id, path_type, span_id, hop_sequence, created_at, origin FROM p54_sp3 WHERE span_id NOT LIKE 'BB-%')""",
    "span_missing": """SELECT count(*) FROM (SELECT path_id, service_id, path_type, span_id, hop_sequence, created_at, origin FROM p54_sp3
                       WHERE span_id NOT LIKE 'BB-%' EXCEPT ALL
                       SELECT path_id, service_id, path_type, span_id, hop_v3, created_at, origin FROM v4_service_path WHERE span_remap_method = 'UNCHANGED_V3')""",
    "unresolved": "SELECT count(*) FROM p54_orient WHERE traverse_dir IS NULL",
    "route_repeat": "SELECT count(*) FROM (SELECT path_id, span_id FROM p54_sp3 WHERE span_id LIKE 'BB-%' GROUP BY 1, 2 HAVING count(*) > 1)",
    "seq_break": f"""SELECT count(*) FROM (SELECT p.traverse_dir, g.segment_seq,
                       lag(g.segment_seq) OVER (PARTITION BY p.path_id, p.hop_v3 ORDER BY p.hop_sequence) AS prv
                     FROM v4_service_path p JOIN {SPAN_V4} g ON g.span_id = p.span_id WHERE p.origin = 'DERIVED')
                     WHERE prv IS NOT NULL AND segment_seq - prv <> CASE WHEN traverse_dir = 'AZ' THEN 1 ELSE -1 END""",
    "hop1_bbs_az": "SELECT count(*) FROM v4_service_path WHERE hop_sequence = 1 AND path_type = 'working' AND span_id LIKE 'BBS-%' AND traverse_dir = 'AZ'",
    "hop1_bbs_za": "SELECT count(*) FROM v4_service_path WHERE hop_sequence = 1 AND path_type = 'working' AND span_id LIKE 'BBS-%' AND traverse_dir = 'ZA'",
    "working_hop1": "SELECT count(*) FROM v4_service_path WHERE hop_sequence = 1 AND path_type = 'working'",
}
g = {k: spark.sql(v).collect()[0][0] for k, v in q.items()}
gate("P54-G1 row count = 1,655,121 = v3 - BB hops + sum(segment_count)", g["n"] == P54_EXPECT_ROWS == g["expect_n"], {k: g[k] for k in ("n", "expect_n")})
gate("P54-G2 paths and services preserved (10,395 / 8,000)",
     g["paths"] == g["paths_v3"] == 10395 and g["svcs"] == g["svcs_v3"] == 8000, {k: g[k] for k in ("paths", "paths_v3", "svcs", "svcs_v3")})
gate("P54-G3 0 BB-* left, 0 span/service FK orphans vs span_v4", g["bb_left"] == 0 and g["span_orphans"] == 0 and g["svc_orphans"] == 0,
     {k: g[k] for k in ("bb_left", "span_orphans", "svc_orphans")})
gate("P54-G4 PK (path_id, hop_sequence) and path_hop_id unique, hops 1..n, NNNN fits",
     g["dup_pk"] == 0 and g["dup_path_hop_id"] == 0 and g["hop_gaps"] == 0 and g["max_hop"] <= 9999,
     {k: g[k] for k in ("dup_pk", "dup_path_hop_id", "hop_gaps", "max_hop")})
gate("P54-G5 SPAN-* rows EXCEPT ALL v3 = 0/0", g["span_extra"] == 0 and g["span_missing"] == 0, {k: g[k] for k in ("span_extra", "span_missing")})
gate("P54-G6 orientation resolved, no route repeats in a path, 0 intra-route seq breaks",
     g["unresolved"] == 0 and g["route_repeat"] == 0 and g["seq_break"] == 0, {k: g[k] for k in ("unresolved", "route_repeat", "seq_break")})
gate("P54-G7 canonical working hop 1 on BBS: 1,143 AZ + 1,253 ZA = 2,396; 1 per path",
     g["hop1_bbs_az"] == 1143 and g["hop1_bbs_za"] == 1253 and g["working_hop1"] == spark.sql("SELECT count(DISTINCT path_id) FROM p54_sp3 WHERE path_type = 'working'").collect()[0][0],
     {k: g[k] for k in ("hop1_bbs_az", "hop1_bbs_za", "working_hop1")})
print(spark.sql("SELECT span_remap_method, traverse_dir, count(DISTINCT path_id, hop_v3) AS v3_hops, count(*) AS v4_rows FROM v4_service_path GROUP BY ALL ORDER BY 1, 2").toPandas().to_string(index=False))

# BUG-QA-132 (independent): proactive_action_v4 span_id (cell 7b) must equal working hop_sequence 1 of the fully expanded
# v4_service_path built here (all hops oriented, renumbered). Needs cell 7b run first in the same session.
if spark.catalog.tableExists("v4_proactive_action"):
    g132 = spark.sql("""WITH h1 AS (SELECT service_id, span_id FROM v4_service_path WHERE path_type = 'working' AND hop_sequence = 1)
      SELECT count(*) AS n,
        count_if(h1.span_id IS NULL) AS no_hop1,
        count_if(h1.span_id IS NOT NULL AND pa.span_id IS DISTINCT FROM h1.span_id) AS diff,
        count_if(pa.span_id LIKE 'BB-%') AS bb_left, count_if(pa.span_id IS NULL) AS null_span,
        count(DISTINCT pa.service_id) AS svcs
      FROM v4_proactive_action pa LEFT JOIN h1 ON h1.service_id = pa.service_id""").collect()[0]
    gate("BUG-QA-132 proactive_action_v4.span_id == service_path_v4 working hop 1 (0 diffs, 0 BB-*, 0 NULL)",
         g132.diff == 0 and g132.no_hop1 == 0 and g132.bb_left == 0 and g132.null_span == 0, g132.asDict())
else:
    print("BUG-QA-132 gate skipped: run cell 7b (P52) first in this session to build v4_proactive_action.")

# Column comments: v3 service_path comments copied verbatim; new columns documented here (Lakelink Fiber naming).
P54_NEW_COMMENTS = {
    "path_hop_id": "Unique hop key: path_id plus 4-digit renumbered hop_sequence (e.g. WP-VC-00001-0003). Lakelink Fiber service path v4.",
    "hop_v3": "hop_sequence of the source row in v3 service_path@11. All BBS-* segments expanded from one v3 BB-* hop share this value.",
    "v3_span_id": "span_id of the source v3 hop. Equals span_id for SPAN-* rows; the retired BB-* route id for expanded BBS-* rows. Count DISTINCT of this to count v3 hops.",
    "sub_ord": "Order of the BBS-* segment within the expanded v3 hop in traversal direction (segment_seq for AZ, segment_count+1-segment_seq for ZA). 1 for SPAN-* rows.",
    "traverse_dir": "Direction the path traverses the backbone route: AZ (site_a to site_z, segment_seq ascending) or ZA. NULL for unchanged SPAN-* rows.",
    "span_remap_method": "How the hop was mapped to v4 spans: UNCHANGED_V3, PREV_HOP_JUNCTION, NEXT_HOP_JUNCTION or NEAREST_ENDPOINT_TO_A_END (single-hop paths).",
}
_cm = {r.col_name: r.comment for r in spark.sql(f"DESCRIBE TABLE {S}.service_path").collect() if r.col_name and not r.col_name.startswith("#")}
P54_COMMENTS = {c: (_cm.get(c) or P54_NEW_COMMENTS.get(c)) for c in spark.table("v4_service_path").columns}
P54_COMMENTS["hop_sequence"] = "1-based hop order along the path after BB-* routes are expanded into BBS-* segments (renumbered; differs from v3 where a BB-* hop precedes it)."
P54_COMMENTS["origin"] = "Row provenance: SYNTHETIC (carried from v3) or DERIVED (BBS-* segment expanded from a v3 BB-* route hop in the v4 build)."
gate("P54-G8 every column has a comment, none mention the legacy brand",
     all(P54_COMMENTS.values()) and not any("lumen" in v.lower() for v in P54_COMMENTS.values()), {k: bool(v) for k, v in P54_COMMENTS.items()})

if WRITE_ENABLED and P54_WRITE_APPROVED:
    assert_v4_target(SP_V4)
    _esc = lambda s: s.replace("'", "\\'")
    _ddl_cols = ",\n  ".join(f"`{f.name}` {f.dataType.simpleString()} COMMENT '{_esc(P54_COMMENTS[f.name])}'" for f in spark.table("v4_service_path").schema)
    spark.sql(f"""CREATE TABLE IF NOT EXISTS {SP_V4} (\n  {_ddl_cols}\n)
      COMMENT 'Lakelink Fiber PRI v4 candidate service path: service_path@11 with BB-* backbone routes expanded into oriented BBS-* segments (P53 spec, P54 writer). Count DISTINCT v3_span_id for hop counts.'
      TBLPROPERTIES ('telco_project.source' = 'P54-DE-SERVICE-PATH-V4-WRITER')""")
    spark.sql(f"INSERT OVERWRITE {SP_V4} SELECT * FROM v4_service_path")
elif spark.catalog.tableExists(SP_V4):
    diff_vs_live("v4_service_path", SP_V4, "path_hop_id", "P54 service_path_v4")
else:
    print(f"P54 dry run: {SP_V4} not built (write-disabled; needs p54_write_approved in the cutover window). Preview rows: {g['n']:,}.")

In [0]:
gates_pdf = pd.DataFrame(GATES)
display(gates_pdf)
failed = gates_pdf[~gates_pdf["pass"]]
assert failed.empty, f"{len(failed)} gate(s) failed: {failed['gate'].tolist()}"
print(f"All {len(gates_pdf)} gates passed (write_enabled={WRITE_ENABLED}).")